# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">6주차 · FM 과 MLP 로 상품 정보를 넣어 보고, 위험도 상한으로 추천을 거르기</mark>

지금까지 모델은 **누가 무엇을 담았는지(거래 기록)만** 봤습니다.

종목이 레버리지인지, 위험도가 몇인지는 **한 번도 보지 않았습니다.**

오늘은 종목에 붙은 정보인 **상품 정보(item feature)** 를 씁니다. 쓰는 방법은 두 가지입니다.

1. **모델에 넣는다** — 상품 정보까지 입력으로 받는 모델 두 개를 직접 학습해 본다
   - **FM** (factorization machine) — 4주차 행렬분해를 넓힌 모델
   - **MLP** (다층 신경망) — 딥러닝의 기본형
2. **위험도 상한을 건다** — 위험도가 투자 성향에 맞지 않는 종목을 추천 목록에서 뺀다

**오늘의 결론을 미리 말하면** — 우리처럼 **데이터가 작을 때는 단순한 모델이 더 낫습니다.** 왜 그런지를 숫자로 확인합니다.

---

### 오늘의 구성 — 50분 (PPT + 노트북)

| 시간 | 어디서 | 하는 일 |
|---|---|---|
| 개념 30분 | PPT | 상품 정보 · 원핫 인코딩 · **FM 의 짝 계산** · MLP 와 층 · **과적합** · DeepFM · 적합성원칙 |
| 실습 20분 | **이 노트북** | 맨 위부터 실행 → 6.5 · 6.6 점수 읽기 → 6.7 비교 표 → 🗣 함께 보기 → 6.9 · 6.10 위험도 상한 |

### 이 노트북 읽는 법 — 🎯 와 📖

- **🎯 수업** — 수업 시간에 셀을 실행하고 **결과 숫자를 함께 읽는** 절입니다
  - 수업에서는 설명 글을 다 읽지 않습니다. 셀 실행과 「결과 읽는 법」만 봅니다
- **📖 혼자 읽기** — 수업에서는 건너뛰고 **복습할 때 읽는** 부분입니다
  - 절 전체가 📖 인 곳(6.2 · 6.3 · 6.4 · 6.8)과, 🎯 절 안의 「왜 배우나 · 배경 지식 · 계산 순서」 블록이 여기에 속합니다
  - 개념은 PPT 에서 설명하고, 이 노트북의 📖 부분이 그 설명을 더 자세히 풀어 줍니다

⏱ **실습을 시작하면 먼저 메뉴의 `런타임 → 모두 실행` 을 눌러 두세요.** 전체 실행에 40초쯤 걸립니다. 그동안 위에서부터 결과를 읽어 내려가면 됩니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/06_item_features.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

**이 노트북이 쓰는 것**

| 무엇 | 쓰임 |
|---|---|
| `data/items.csv` | 종목 100개 — 이름 · 섹터 · 자산군 · **위험도** · 테마 |
| `data/users.csv` | 투자자 300명 — **투자 성향**(안정형 · 성장형 · 공격형) |
| `data/interactions.csv` | 거래 기록 8,086건 |
| `recsys.py` | 데이터 읽기 · 시간순 분할 · 채점 · 사용자-아이템 행렬 · **오늘 새로: 학습 데이터 만들기 · FM** |

⚠ **지난주 노트북을 먼저 돌리지 않아도 됩니다.** 이 노트북 하나로 끝납니다.

⚠ **따로 설치할 것이 없습니다.** 오늘 쓰는 도구(numpy · pandas · scikit-learn)는 Colab 에 처음부터 들어 있습니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말 열 개</mark>

**1\~2번은 재료, 3\~4번은 모델, 5\~7번은 학습, 8번은 모델의 계보, 9\~10번은 금융 규칙입니다.**

새 말이 많습니다. 여기서 다 외우지 않아도 됩니다. **노트북에서 그 말이 처음 쓰이는 절에서 예와 함께 다시 설명합니다.** 괄호 안이 그 절입니다.

#### 재료

1. **상품 정보 (item feature)** — 거래 기록이 아니라 **종목 자체에 붙은 정보** (6.3)
   - 오늘 쓰는 것은 `items.csv` 의 열(column) 네 개 — 섹터 · 자산군 · 위험도 · 테마
   - 2주차의 **피처(feature)** 와 같은 말입니다. 그때는 사람에게 붙은 피처(주력 섹터 · 평균 위험도)였고, 오늘은 종목에 붙은 피처입니다

2. **원핫 인코딩 (one-hot encoding)** — 글자로 된 범주를 **「그 범주면 1, 아니면 0」인 열 여러 개**로 바꾸는 방법 (6.3)
   - 이유 — 글자(`레버리지`)로는 곱하거나 더할 수 없습니다
   - 범주 하나마다 열이 하나 생기고, 한 줄에서 **1 이 들어가는 열은 하나뿐**이라서 「원핫(one-hot)」이라고 부릅니다
   - 예 — 종목 세 개의 섹터를 원핫 인코딩하면 아래 ① → ② 처럼 바뀝니다

① **원래** — 섹터가 글자로 적혀 있습니다.

| 종목 | 섹터 |
|---|---|
| KODEX 레버리지 | 레버리지 |
| 삼성생명 | 금융 |
| Procter & Gamble | 소비재 |

② **원핫 인코딩 후** — 섹터 하나마다 열을 하나 만들고, 그 섹터면 1, 아니면 0 을 적습니다.

| 종목 | 레버리지 | 금융 | 소비재 |
|---|---|---|---|
| KODEX 레버리지 | **1** | 0 | 0 |
| 삼성생명 | 0 | **1** | 0 |
| Procter & Gamble | 0 | 0 | **1** |


#### 모델

3. **FM (factorization machine, 팩토리제이션 머신)** — **1 이 들어간 열들을 둘씩 짝지어**, 짝마다 잠재 요인의 내적을 구해 모두 더한 값을 점수로 쓰는 모델 (6.4 \~ 6.5)
   - 4주차 행렬분해는 「사용자 × 종목」 **한 짝**의 내적만 봤습니다
   - FM 은 「사용자 × 위험도」, 「종목 × 섹터」 같은 **모든 짝**을 봅니다. 그래서 상품 정보를 넣을 수 있습니다

4. **MLP (multi-layer perceptron, 다층 신경망)** — 입력에 가중치를 곱해 더하는 계산을 **여러 번 거쳐** 확률을 내는 모델 (6.6)
   - 그 계산 한 번을 **층(layer)** 이라고 합니다. 층이 많을수록 「깊다(deep)」고 하고, 이것이 **딥러닝(deep learning)** 이라는 이름의 뜻입니다
   - FM 은 「짝의 내적」이라는 계산 모양이 정해져 있고, MLP 는 어떤 계산을 할지 **데이터를 보고 스스로 정합니다**

#### 학습

5. **네거티브 샘플링 (negative sampling)** — 사용자가 **담지 않은 종목**을 골라 「0 (안 담음)」 예시로 학습 데이터에 넣는 것 (6.5)
   - 거래 기록에는 「담았다(1)」만 있습니다. 1 만 보고 배우면 모델은 「무엇이든 담는다」고 배웁니다

6. **에폭 (epoch)** — 학습 데이터를 처음부터 끝까지 **한 번 다 보는 것** (6.5)
   - 모델은 한 번 보고 끝나지 않고, 여러 번 돌면서 숫자를 조금씩 고칩니다. 「에폭 80」 = 80번 돌았다

7. **과적합 (overfitting)** — 모델이 **학습 데이터를 외워 버려서**, 학습 데이터는 잘 맞히는데 **새 데이터는 못 맞히는** 상태 (6.7)
   - 모델이 복잡할수록(고칠 숫자가 많을수록), 데이터가 적을수록 잘 생깁니다

#### 모델의 계보

8. **Wide&Deep · DeepFM** — 단순한 모델(Wide 또는 FM)과 MLP(Deep)를 **한 모델로 묶은 것** (6.8)
   - 구글 · 화웨이가 데이터가 아주 많은 서비스(앱 스토어 · 광고)를 위해 만들었습니다
   - 오늘은 직접 돌리지 않고 **개념만** 봅니다. 왜 안 돌리는지도 6.7 의 숫자가 알려 줍니다

#### 금융 규칙

9. **적합성원칙 (suitability rule)** — 금융 회사는 **고객의 투자 성향에 맞지 않는 상품을 권유하면 안 된다**는 원칙 (6.9)
   - 금융소비자보호법 제17조에 있습니다
   - **투자성향**은 고객이 가입할 때 설문으로 정합니다 — 실제는 5단계, 우리 데이터는 안정형 · 성장형 · 공격형 3단계
   - 실제 **상품 위험등급**은 6단계이고 **1등급이 가장 위험**합니다. 우리 데이터의 위험도(1\~5)는 5가 가장 위험해서 방향이 반대입니다

10. **위험도 상한 (risk cap)** — 투자 성향마다 정한 **추천해도 되는 위험도의 최댓값** (6.10)
    - 이 값을 넘는 종목은 추천 목록에서 뺍니다
    - 모델이 만든 목록에서 규칙으로 종목을 빼는 것을 **후처리 필터링(post-filtering)** 이라고 합니다

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 — 데이터 읽기와 시간순 분할</mark>

**여기서 하는 일** — 데이터를 읽고, 3주차부터 쓰던 **시간순 분할**로 학습 구간과 채점 구간을 나눕니다.

**채점 기준이 같아야** 오늘 숫자를 4주차 **0.3918** 과 비교할 수 있습니다.

In [2]:
# 이 셀에서 하는 일 — 도구를 불러오고, 데이터를 읽고, 시간순으로 나눈다
# 왜 하나 — 지난주와 같은 기준으로 채점해야 0.3918 과 비교할 수 있다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 단계 안쪽 폴더(notebooks)도 찾게 한다

import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import pandas as pd                         # 표를 다루는 도구. pd 라고 부른다
import recsys                               # 이 수업용 도구 모음 — 읽기·분할·채점이 들어 있다

items, users, interactions = recsys.load()                  # 종목·투자자·거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 3주차부터 쓰던 시간순 분할

종목이름 = items.set_index("item_id")["name"].to_dict()        # to_dict = 딕셔너리로. {종목 번호: 이름}
이미_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # groupby = 사람별로 묶는다
전체_인기순위 = list(train["item_id"].value_counts().index)   # value_counts = 같은 값이 몇 번인지 센다

print(f"학습 {len(train):,}건 · 채점 {len(test):,}건 · 자른 날짜 {기준시점.date()}")

학습 7,095건 · 채점 991건 · 자른 날짜 2026-07-30


**결과 읽는 법**

- 거래 기록 8,086건을 2026-07-30 을 기준으로 둘로 나눈 결과입니다.
  - **학습 7,095건**은 자른 날짜 앞쪽 기록입니다. 모델이 배우는 데 씁니다.
  - **채점 991건**은 자른 날짜 뒤쪽 기록입니다. 추천이 맞았는지 확인하는 데 씁니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.1 준비 — 4주차 행렬분해를 다시 만든다</mark>

**6.1 에서 하는 일** — 4주차의 **사용자 요인(user factor, 285 × 8)** 과 **아이템 요인(item factor, 8 × 100)** 을 다시 만들고, 종목 100개 전부에 점수를 매기는 4주차 추천을 다시 돌립니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 오늘 FM · MLP 두 모델을 새로 만듭니다. 새 모델이 좋은지 나쁜지 판단하려면 **비교할 기준**이 있어야 합니다.
  - 그 기준이 지금까지 가장 점수가 높았던 4주차 행렬분해(0.3918)입니다.

**왜 중요한가**
- 점수 하나만 보면 좋은지 알 수 없습니다.
  - 예 — FM 이 0.35 가 나왔다면, 4주차보다 낮으므로 「상품 정보를 넣었더니 오히려 나빠졌다」는 뜻입니다.
- 비교가 공정하려면 **같은 데이터 · 같은 분할 · 같은 채점**이어야 합니다.
  - 그래서 6.1 에서 4주차 모델을 그대로 다시 만들고, 0.3918 이 똑같이 나오는지부터 확인합니다.

**알아 둘 배경 지식**
- **사용자-아이템 행렬** (4주차) — 투자자 285명 × 종목 100개 표. 담았으면 1, 아니면 0
- **잠재 요인 (latent factor)** (4주차) — 거래 기록에서 데이터가 스스로 찾은 취향의 기준. 사람이 이름을 붙이지 않은 숫자 8개
- **내적 (dot product)** — 두 숫자 목록을 같은 자리끼리 곱해서 모두 더한 값
  - 예 — `[1, 2]` 와 `[3, 4]` 의 내적 = 1×3 + 2×4 = 11
- **Recall@10** (1주차) — 다음 달 실제로 담은 종목 가운데, 추천 10개 안에 들어 있던 비율
- **시간순 분할** (3주차) — 과거 기록으로 배우고 마지막 1개월을 맞히게 해서 채점한다

**어떻게 계산하나 — 4주차 추천의 순서**
1. 학습 구간 거래 기록으로 285 × 100 사용자-아이템 행렬을 만든다
2. TruncatedSVD 가 이 행렬을 **사용자 요인(285 × 8)** 과 **아이템 요인(8 × 100)** 두 개로 쪼갠다
   - 둘을 다시 곱하면 원래 행렬과 최대한 비슷해지도록 숫자를 정한다
3. 한 투자자의 숫자 8개와 한 종목의 숫자 8개를 내적한다 → 그 종목의 점수
   - 종목 100개 모두에 이렇게 점수를 매긴다
4. 점수가 높은 순으로 늘어놓고, 이미 담은 종목을 빼고 위에서 10개를 추천한다

---

#### 자세히

**1. 왜 5주차가 아니라 4주차와 비교하나**
- 오늘 비교할 대상은 **4주차 0.3918** 입니다.
- 5주차 2단계 추천까지 섞으면, 점수가 바뀐 것이 상품 정보 때문인지 후보 40개 때문인지 가릴 수 없습니다.
- **한 번에 하나만 바꿔야** 비교가 정확합니다.

**2. 함수 두 개를 만듭니다**
- `점수순으로_종목_정렬` — 종목 100개의 점수를 받아 **점수가 높은 종목부터 종목 번호를 차례로 담은 목록**을 돌려줍니다. 오늘 만드는 추천이 모두 이 함수를 씁니다 — 4주차 · FM · MLP · 위험도 상한.
- `지난주_추천` — 4주차 추천과 같습니다. 사용자 요인 한 줄 × 아이템 요인 = 종목 100개 점수.

**3. 변수 이름**
- 4·5주차 노트북의 `사람표` · `종목표` 를 오늘은 공식 용어 그대로 `사용자_요인` · `아이템_요인` 이라고 부릅니다.
- 285 × 100 짜리 「담았으면 1, 아니면 0」 표는 **사용자-아이템 행렬(user-item matrix)** 이므로 `상호작용_행렬` 이라고 부릅니다.

In [3]:
# 이 셀에서 하는 일 — 4주차와 똑같이 사용자-아이템 행렬을 만들고 잠재 요인 8개로 쪼갠다
# 왜 하나 — 오늘 쓰는 모델이 지난주 것과 같아야 상품 정보의 효과만 볼 수 있다

from sklearn.decomposition import TruncatedSVD      # 4주차에 쓴 행렬분해 도구

상호작용_행렬 = recsys.interaction_matrix(train, items)        # 사용자-아이템 행렬 — 담았으면 1, 아니면 0
모델 = TruncatedSVD(n_components=8, random_state=42)   # 잠재 요인 8개 — 지난주와 같은 설정
사용자_요인 = 모델.fit_transform(상호작용_행렬)                      # 사용자 요인(user factor) · 285 × 8
아이템_요인 = 모델.components_                            # 아이템 요인(item factor) · 8 × 100

사용자_목록 = list(상호작용_행렬.index)                         # 행렬의 행 이름(투자자 번호) 목록
아이템_목록 = list(상호작용_행렬.columns)                            # 행렬의 열 이름(종목 번호) 목록

print(f"사용자_요인 {사용자_요인.shape} · 아이템_요인 {아이템_요인.shape}   ← 지난주와 같은 크기")

사용자_요인 (285, 8) · 아이템_요인 (8, 100)   ← 지난주와 같은 크기


**결과 읽는 법**

- 4주차와 같은 크기의 요인 두 개가 만들어졌습니다.
  - `사용자_요인 (285, 8)` 은 투자자 285명 × 잠재 요인 8개입니다.
  - `아이템_요인 (8, 100)` 은 잠재 요인 8개 × 종목 100개입니다.

In [4]:
# 이 셀에서 하는 일 — 종목 100개의 점수를 받아, 점수가 큰 종목부터 늘어놓는 함수를 만든다
# 왜 하나 — 오늘 만드는 추천이 모두 「점수가 큰 종목부터 늘어놓기」를 한다

def 점수순으로_종목_정렬(점수):                       # def = 함수를 만든다. 종목 100개의 점수를 받는다
    """종목 100개의 점수를 받아, 점수가 큰 종목부터 종목 번호를 늘어놓은 목록을 돌려준다."""
    정렬된_종목 = []                                # 종목 번호를 점수 순서대로 담을 빈 목록
    for 자리 in np.argsort(-점수):                  # argsort(-점수) = 점수가 큰 것부터 그 자리 번호
        정렬된_종목.append(아이템_목록[자리])          # 자리 번호(0~99) → 종목 번호(I001~I100)
    return 정렬된_종목                              # 점수 1등 종목부터 100등 종목까지

In [5]:
# 이 셀에서 하는 일 — 방금 만든 함수를 U0001 의 점수에 한 번 써 본다
# 왜 하나 — 함수가 무엇을 돌려주는지 눈으로 확인한다

U0001_점수 = 사용자_요인[사용자_목록.index("U0001")] @ 아이템_요인   # U0001 의 종목 100개 점수 (4주차 계산)
정렬된_종목 = 점수순으로_종목_정렬(U0001_점수)                     # 점수가 큰 종목부터 늘어놓는다

순위 = 1                                                          # 몇 등인지 세는 카운터
for 번호 in 정렬된_종목[:3]:                                       # [:3] = 앞에서 3개만
    print(f"{순위}등 {번호} · 점수 {U0001_점수[아이템_목록.index(번호)]:.3f} · {종목이름[번호]}")
    순위 = 순위 + 1                                               # 다음 등수로
print("※ 세 종목 모두 U0001 이 이미 담은 종목입니다. 이것을 빼는 일은 다음 셀의 recsys.take 가 합니다.")

1등 I007 · 점수 0.973 · KODEX 국고채30년액티브
2등 I006 · 점수 0.909 · TIGER 리츠부동산인프라
3등 I019 · 점수 0.794 · Procter & Gamble
※ 세 종목 모두 U0001 이 이미 담은 종목입니다. 이것을 빼는 일은 다음 셀의 recsys.take 가 합니다.


**결과 읽는 법**

- U0001 의 4주차 점수가 높은 종목 3개입니다.
  - 한 줄은 「등수 · 종목 번호 · 점수 · 종목 이름」 순서입니다.
  - 1등은 I007 KODEX 국고채30년액티브이고, 점수는 **0.973** 입니다.
- 마지막 줄 ※ 는 이 세 종목이 U0001 이 **이미 담은 종목**이라는 뜻입니다.
  - 그래서 이 세 종목은 실제 추천 목록에는 들어가지 않습니다.

In [6]:
# 이 셀에서 하는 일 — 4주차 추천 함수를 다시 만들고 U0001 에게 한 번 돌려 본다
# 왜 하나 — 오늘 모든 비교의 출발점이다

def 지난주_추천(사람, 이미):                          # def = 함수를 만든다
    """4주차 방식 — 종목 100개 전부에 점수를 매겨 높은 순으로 10개."""
    if 사람 not in 사용자_목록:                      # 사용자-아이템 행렬에 없는 신규 투자자는
        return recsys.take(전체_인기순위, 이미)      # 전체 인기 목록으로 준다
    점수 = 사용자_요인[사용자_목록.index(사람)] @ 아이템_요인   # @ = 행렬 곱. 사용자 요인 한 줄 × 아이템 요인
    return recsys.take(점수순으로_종목_정렬(점수), 이미)        # take = 이미 담은 것을 빼고 위에서 10개

print("U0001 에게 줄 추천 1등 :", 종목이름[지난주_추천("U0001", 이미_담은것["U0001"])[0]])

U0001 에게 줄 추천 1등 : TIGER 미국배당다우존스


**결과 읽는 법**

- U0001 이 이미 담은 종목을 빼고 남은 종목 가운데 1등입니다.
  - 바로 위 셀의 1등(KODEX 국고채30년액티브)이 빠지고 **TIGER 미국배당다우존스**가 1등이 되었습니다.

In [7]:
# 이 셀에서 하는 일 — 4주차 추천을 3주차와 같은 기준으로 채점한다
# 왜 하나 — 0.3918 이 나와야 지난주 모델을 제대로 다시 만든 것이다

지난주_점수, 인원 = recsys.recall_at_k(지난주_추천, train, test)   # 3주차부터 쓰던 채점 함수

print(f"4주차 추천 — Recall@10 = {지난주_점수:.4f}   (채점 대상 {인원}명)")

4주차 추천 — Recall@10 = 0.3918   (채점 대상 266명)


**결과 읽는 법**

- 6.1 에서 다시 만든 4주차 추천의 채점 결과입니다.
  - Recall@10 은 **0.3918** 로, 4주차에 나온 값과 같습니다.
  - 채점 대상은 **266명**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.2 4주차 추천의 위험도 — 안정형 U0001 의 10개</mark>

> 📖 **혼자 읽기** — 6.9 · 6.10 에서 같은 문제를 300명 전체 숫자로 다시 봅니다.

**6.2 에서 하는 일**
- 6.1 에서 다시 만든 4주차 모델로 **U0001 에게 줄 추천 10개**를 뽑습니다.
- 각 종목 옆에 그 종목의 **위험도(1 \~ 5)** 를 적어 봅니다.
- U0001 은 **안정형** 투자자입니다. 추천 10개 안에 위험도가 높은 종목이 섞여 있는지 확인합니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 지금까지는 **점수(Recall@10)가 높으면 좋은 추천**이라고 봤습니다.
- 금융에서는 그것만으로 부족합니다. 고객이 담을 만한 종목이라도 **고객의 투자 성향에 맞지 않으면 권하면 안 됩니다.**
- 6.2 는 「점수가 가장 높은 4주차 모델이 이 기준을 지키는가」를 한 사람으로 먼저 확인하는 절입니다.

**왜 중요한가**
- 추천 시스템은 수백 · 수천 명에게 **자동으로** 상품을 보여 줍니다.
  - 상담 직원 한 명의 실수는 고객 한 명에게 그치지만, 모델의 실수는 같은 방식으로 수백 건이 됩니다.
- 성향에 맞지 않는 권유는 **적합성원칙(금융소비자보호법 제17조)** 에 어긋납니다.
  - 고객이 손실을 보면 회사가 책임을 질 수 있고, 금융당국의 제재 대상이 될 수 있습니다.

**알아 둘 배경 지식**
- **협업 필터링** — 「나와 비슷하게 담은 사람들이 담은 종목」을 추천하는 방식. 4주차 행렬분해가 여기에 속합니다
  - 입력이 거래 기록뿐이라서, 종목이 얼마나 위험한지 · 고객이 어떤 성향인지 **모델은 알 방법이 없습니다**
- **딕셔너리** — `{키: 값}` 모양의 자료. 키로 값을 바로 찾습니다
  - 예 — `위험도["I079"]` → 5

**어떻게 계산하나**
1. 6.1 의 4주차 모델로 U0001 의 추천 10개를 뽑는다
2. 종목 번호마다 `위험도` 딕셔너리에서 위험도를 찾아 옆에 적는다
3. 위험도가 4 이상이면 ⚠ 를 붙인다 — 안정형에게는 높은 위험도

---

#### 자세히

**왜 확인하나**
- 4주차 모델은 **거래 기록만** 보고 추천했습니다.
  - 종목이 얼마나 위험한지(위험도)는 보지 않았습니다.
  - 투자자가 어떤 투자 성향인지도 보지 않았습니다.
- 그래서 안정형 투자자에게 위험도가 높은 종목을 추천했을 수 있습니다. 실제로 그런지 출력으로 확인합니다.

**1. 위험도란**
- `items.csv` 의 `risk_level` 열입니다. 종목마다 1 \~ 5 중 하나가 적혀 있습니다.
  - 1 = 가장 낮음 (예: KODEX 단기채권)
  - 5 = 가장 높음 (예: KODEX 레버리지)
- 종목 100개 중 위험도 4 는 28개, 5 는 20개입니다.

**2. 투자 성향이란**
- `users.csv` 의 `persona` 열입니다. 투자자마다 안정형 · 성장형 · 공격형 중 하나가 적혀 있습니다.
  - 안정형 = 원금 손실을 되도록 피하려는 투자자
  - 성장형 = 어느 정도 손실을 감수하고 수익을 노리는 투자자
  - 공격형 = 큰 손실 가능성을 감수하고 높은 수익을 노리는 투자자
- 우리 데이터에는 세 성향이 100명씩 있습니다.
- 실제 금융 회사는 고객이 계좌를 열 때 **투자성향 설문**으로 이 값을 알아냅니다.

**3. 2주차의 세그먼트와 무엇이 다른가**
- 2주차에는 성향을 **거래 기록으로 짐작**했습니다.
  - 주력 섹터와 평균 위험도로 세그먼트(예: `레버리지|4`)를 만들었습니다.
- 오늘은 **고객이 설문에 직접 답한 성향**(`persona`)을 씁니다.
  - 「이 고객에게 위험도 몇까지 추천해도 되나」는 짐작으로 정하면 안 됩니다. 고객이 직접 밝힌 성향을 따라야 합니다.

**4. 아래 코드가 하는 일 (3단계)**
1. `성향` 딕셔너리를 만든다 — 투자자 번호 → 투자 성향 (예: U0001 → 안정형)
2. `위험도` 딕셔너리를 만든다 — 종목 번호 → 위험도 (예: I079 → 5)
3. U0001 의 추천 10개를 한 줄씩 출력한다
   - 위험도가 **4 이상**이면 줄 앞에 ⚠ 를 붙인다

**5. 출력에서 찾을 것**
- ⚠ 가 붙은 줄이 몇 개인지 세어 보세요.

In [8]:
# 이 셀에서 하는 일 — U0001 의 추천 10개에 위험도를 붙여 본다
# 왜 하나 — 지난주 모델이 투자 성향을 전혀 보지 않는다는 것을 눈으로 확인한다

성향 = users.set_index("user_id")["persona"].to_dict()       # 딕셔너리 {투자자 번호: 안정형/성장형/공격형}
위험도 = items.set_index("item_id")["risk_level"].to_dict()   # 딕셔너리 {종목 번호: 1~5}
print(f"U0001 의 투자 성향 : {성향['U0001']}")

순위 = 1                                                      # 몇 등인지 세는 카운터
for 번호 in 지난주_추천("U0001", 이미_담은것["U0001"]):      # 지난주 모델이 준 10개
    표시 = "⚠" if 위험도[번호] >= 4 else " "                 # >= 크거나 같다. 4 이상이면 ⚠
    print(f"{순위:2d}. 위험도 {위험도[번호]} {표시} {종목이름[번호]}")   # :2d = 두 자리로 오른쪽 정렬
    순위 = 순위 + 1                                           # 다음 등수로

U0001 의 투자 성향 : 안정형
 1. 위험도 2   TIGER 미국배당다우존스
 2. 위험도 5 ⚠ TIGER 2차전지테마
 3. 위험도 3   Vanguard FTSE Developed Markets ETF
 4. 위험도 2   TIGER 은행고배당플러스TOP10
 5. 위험도 1   KODEX 단기채권
 6. 위험도 4 ⚠ 크래프톤
 7. 위험도 4 ⚠ Taiwan Semiconductor
 8. 위험도 2   Verizon Communications
 9. 위험도 2   Johnson & Johnson
10. 위험도 4 ⚠ Advanced Micro Devices


**결과 읽는 법**

- ⚠ 가 **4개**입니다. 안정형 U0001 에게 위험도 4 \~ 5 종목이 10개 중 4개 들어갔습니다.
  - 2등 TIGER 2차전지테마 — 위험도 5
  - 6등 크래프톤 · 7등 Taiwan Semiconductor · 10등 Advanced Micro Devices — 위험도 4

- 왜 이런 추천이 나왔나
  - 4주차 모델은 **협업 필터링**입니다. U0001 과 **거래 기록이 비슷한 다른 투자자들**이 담은 종목을 추천합니다.
  - 그 투자자 가운데 위험도가 높은 종목을 담은 사람이 있으면, 그 종목이 U0001 의 추천에도 들어옵니다.
  - 모델은 종목의 위험도도, U0001 의 투자 성향도 모릅니다. 거래 기록만 입력으로 받았기 때문입니다.

- 그래서 오늘 두 가지를 해 봅니다
  - 6.3 \~ 6.8 — 상품 정보를 **모델에 넣으면**(FM · MLP) 추천이 나아지는지 본다
  - 6.9 \~ 6.10 — 투자 성향에 맞지 않는 위험도의 종목을 추천 목록에서 **뺀다**

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.3 원핫 인코딩 — 상품 정보를 0/1 열로 펼치기</mark>

> 📖 **혼자 읽기** — 수업에서는 맨 위 「오늘 나오는 말」 2번의 표 한 장으로 설명합니다.

**6.3 에서 하는 일** — 종목마다 네 가지 상품 정보를 **0과 1로 된 열**로 바꿉니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 오늘 쓰는 FM 과 MLP 는 **숫자만 입력으로 받습니다.**
- 그런데 상품 정보는 `레버리지` · `ETF` · `배당` 처럼 **글자**입니다. 모델에 넣으려면 먼저 숫자로 바꿔야 합니다.
- 원핫 인코딩이 그 변환 방법입니다.

**왜 중요한가**
- 실제 데이터에는 이런 글자 정보가 매우 많습니다 — 성별 · 지역 · 상품 종류 · 가입 채널 · 업종 …
- 원핫 인코딩은 이런 정보를 모델에 넣는 **가장 기본적인 방법**이라, 대부분의 머신러닝 모델 앞 단계에 들어갑니다.
- 6.5 에서는 상품 정보뿐 아니라 **투자자 번호와 종목 번호도** 원핫 인코딩합니다. FM 과 MLP 의 입력 전체가 원핫 인코딩으로 만들어집니다.

**알아 둘 배경 지식**
- **범주형 데이터 (categorical data)** — 크기 · 순서가 없는 이름표 같은 값
  - 예 — 섹터(`레버리지` · `금융`), 자산군(`ETF` · `주식`)
  - 「금융이 레버리지보다 크다」는 말이 성립하지 않습니다
- **수치형 데이터 (numerical data)** — 크기를 비교하고 더하고 평균을 낼 수 있는 값
  - 예 — 가격, 거래 횟수
- 모델은 입력을 **수치형으로** 다룹니다. 그래서 범주형을 수치형으로 그냥 바꾸면(레버리지 = 1, 금융 = 2) 없는 크기가 생깁니다. 아래 3번에서 자세히 봅니다.

**어떻게 계산하나**
1. 한 열(예: 섹터)에 나오는 값을 **모두 모읍니다** — 섹터는 25가지
2. 값마다 **새 열을 하나씩** 만듭니다 — 열 25개
3. 종목마다, **자기 값의 열에만 1**, 나머지 열에는 0 을 적습니다
4. 자산군 · 위험도 · 테마에도 똑같이 하고 옆으로 붙입니다 — 25 + 2 + 5 + 9 = 41열

---

#### 자세히

**1. 왜 바꾸나**
- `레버리지` · `금융` 같은 글자로는 곱셈도 평균도 못 합니다.
- 6.4 \~ 6.6 의 FM 과 MLP 는 **숫자만 입력으로 받습니다.** 상품 정보를 넣으려면 먼저 숫자로 바꿔야 합니다.

**2. 예시 — 종목 세 개의 섹터만 펼쳐 보면**

①과 ② 두 표는 맨 위 **「오늘 나오는 말」 2번**에 있습니다. 그 표를 보면서 아래를 읽어 주세요.

③ **읽는 법**
- 맨 윗줄 `레버리지 · 금융 · 소비재` 는 **열 이름**, 왼쪽 종목 이름은 **행 이름**입니다. 둘 다 이름일 뿐이라 계산에는 쓰지 않습니다.
- **계산에 쓰는 값은 가운데의 0 과 1 뿐**입니다.
- 한 줄에 1 이 **딱 하나만** 있습니다. 그래서 「원핫(one-hot)」입니다.
- **바로 아래 첫 셀이 이 두 표를 그대로 만들고, 두 번째 셀은 값(0 과 1)만 꺼내 보여 줍니다.**

**3. 왜 1 · 2 · 3 으로 번호를 붙이지 않나**
- 레버리지 = 1, 금융 = 2, 소비재 = 3 으로 적으면 **「금융은 레버리지의 두 배」, 「소비재와 레버리지의 평균이 금융」** 같은 없는 뜻이 생깁니다.
- 섹터에는 크기 순서가 없습니다. 원핫 인코딩을 하면 **어느 두 범주든 서로 거리가 같습니다**.

**4. 네 가지를 다 펼치면**
- 섹터 25 + 자산군 2 + 위험도 5 + 테마 9 = **41개 열**
- 열 이름 앞에 원래 열 이름이 붙습니다 — `sector_레버리지` · `theme_레버리지`. 「레버리지」가 섹터에도 테마에도 있어서, 붙이지 않으면 두 열의 이름이 같아집니다.
- 종목마다 정보가 네 가지이므로, 한 줄에 **1 이 정확히 4개**입니다(아래 출력의 I079).
- 위험도는 숫자지만 **범주로 다룹니다.** 「위험도 4 인가 아닌가」만 봅니다.

**5. 어떤 함수를 쓰나**
- `pd.get_dummies` — pandas 의 원핫 인코딩 함수입니다. 「더미(dummy) 변수」는 0/1 열을 가리키는 통계 용어입니다.

In [9]:
# 이 셀에서 하는 일 — 종목 세 개의 섹터를 원핫 인코딩해 본다 (위 ① → ②)
# 왜 하나 — 100개 × 41열로 가기 전에, 작은 예시로 무엇이 바뀌는지 먼저 본다

섹터_예시 = pd.Series(["레버리지", "금융", "소비재"],
                      index=["KODEX 레버리지", "삼성생명", "Procter & Gamble"])   # Series = 이름이 붙은 값 목록
print(섹터_예시.to_frame("섹터"))                         # ① 원래 — to_frame = 표 모양으로 보여 준다
print()

원핫_예시 = pd.get_dummies(섹터_예시).astype(int)          # get_dummies = 원핫 인코딩. astype(int) = True/False → 1/0
원핫_예시 = 원핫_예시[["레버리지", "금융", "소비재"]]        # 열 순서를 위 설명 표와 같게 맞춘다
print(원핫_예시)                                          # ② 원핫 인코딩 후

                    섹터
KODEX 레버리지        레버리지
삼성생명                금융
Procter & Gamble   소비재

                  레버리지  금융  소비재
KODEX 레버리지           1   0    0
삼성생명                 0   1    0
Procter & Gamble     0   0    1


**결과 읽는 법**

- 위쪽 표는 원래 모양(①)입니다.
  - 종목 세 개의 섹터가 글자로 적혀 있습니다.
- 아래쪽 표는 원핫 인코딩 후(②)입니다.
  - 섹터마다 열이 하나씩 생기고, 그 섹터인 칸만 **1** 입니다.

In [10]:
# 이 셀에서 하는 일 — 열 이름 · 행 이름을 떼고 값만 꺼내 본다
# 왜 하나 — 표에 보이는 글자는 열 이름 · 행 이름일 뿐, 계산에 쓰는 것은 0 과 1 뿐이라는 것을 확인한다

print(원핫_예시.values)                                   # .values = 열 이름 · 행 이름을 떼고 값만

[[1 0 0]
 [0 1 0]
 [0 0 1]]


**결과 읽는 법**

- 바로 위 아래쪽 표에서 열 이름과 행 이름을 떼고 0 과 1 만 남긴 것입니다.
  - 세 줄은 위에서부터 KODEX 레버리지 · 삼성생명 · Procter & Gamble 입니다.
  - 세 칸은 왼쪽부터 레버리지 · 금융 · 소비재 열입니다.

In [11]:
# 이 셀에서 하는 일 — 종목 100개의 상품 정보 네 가지를 0/1 열 41개로 펼친다
# 왜 하나 — 글자로는 계산을 못 한다. 0/1 이어야 평균도 곱셈도 된다

정보_열 = ["sector", "asset_class", "risk_level", "theme"]          # 섹터 · 자산군 · 위험도 · 테마
종목_정보 = items.set_index("item_id")[정보_열].astype(str)          # astype(str) = 위험도 숫자도 글자로 본다
아이템_피처 = pd.get_dummies(종목_정보).astype(int)                       # get_dummies = 원핫 인코딩. 결과가 아이템 피처 행렬(item feature matrix)
아이템_피처 = 아이템_피처.loc[아이템_목록]                                          # loc = 종목 순서를 사용자-아이템 행렬과 맞춘다
print(f"아이템_피처 {아이템_피처.shape}   ← 종목 100개 × 열 41개")

값이_1인_열 = []                                                         # 1 인 열 이름을 담을 빈 목록
for 열 in 아이템_피처.columns:                                            # 41개 열을 하나씩 본다
    if 아이템_피처.loc["I079", 열] == 1:                                   # == 같다. 1 인 열만 모은다
        값이_1인_열.append(열)                                            # append = 목록 끝에 붙인다
print(f"I079 {종목이름['I079']} → 1 인 열 : {값이_1인_열}")

아이템_피처 (100, 41)   ← 종목 100개 × 열 41개
I079 KODEX 레버리지 → 1 인 열 : ['sector_레버리지', 'asset_class_ETF', 'risk_level_5', 'theme_레버리지']


**결과 읽는 법**

- 첫 줄 `(100, 41)` 은 아이템 피처 행렬의 크기입니다.
  - 종목 100개 × 0/1 열 41개입니다.
- 둘째 줄은 I079 KODEX 레버리지 줄에서 값이 1 인 열 이름입니다.
  - 섹터 · 자산군 · 위험도 · 테마에서 하나씩, 모두 **4개**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.4 FM 의 생각 — 1 인 열들을 둘씩 짝지어 내적을 더한다</mark>

> 📖 **PPT 에서 같은 계산을 설명합니다.** 이 절은 그 계산을 직접 돌려 보는 복습용입니다.

**6.4 에서 하는 일** — FM 이 점수를 어떻게 계산하는지, **숫자 몇 개로 직접** 따라가 봅니다. 학습은 6.5 에서 합니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 4주차 행렬분해는 **투자자 번호와 종목 번호만** 입력으로 받습니다. 상품 정보를 넣을 자리가 없습니다.
- FM 은 행렬분해의 계산 방식(내적)을 그대로 쓰면서 **상품 정보도 함께 넣을 수 있게** 넓힌 모델입니다.
- 노션 계획표의 6주차 주제 「딥러닝 추천 (FM · DeepFM)」의 첫 번째 모델입니다.

**왜 중요한가**
- FM 은 광고 클릭 예측 · 상품 추천 같은 실무에서 오래 쓰인 **기본 모델**입니다.
- 거래가 적은 종목에도 점수를 매길 수 있습니다.
  - 새로 나온 종목은 거래 기록이 없어서 행렬분해로는 점수를 매기기 어렵습니다 (3주차의 신규 고객 문제와 같은 종류).
  - FM 은 「위험도 5」 · 「반도체」 같은 상품 정보 열이 이미 학습되어 있어서 그 정보로 점수를 냅니다.
- 6.8 의 **DeepFM** 은 FM 에 MLP 를 붙인 모델입니다. FM 을 알아야 DeepFM 이 무엇인지 이해할 수 있습니다.

**알아 둘 배경 지식**
- **내적** (6.1) — 같은 자리끼리 곱해 더한 값
  - 두 목록의 부호가 같은 방향이면 **큰 양수**, 반대 방향이면 **음수**가 나옵니다
  - 그래서 「이 둘은 잘 어울린다(양수) / 안 어울린다(음수)」를 나타내는 데 씁니다
- **잠재 요인** (4주차) — 4주차에는 투자자와 종목에만 붙어 있었습니다. FM 은 **열마다** 붙입니다 — `위험도 5` 열에도, `반도체` 열에도
- **원핫 인코딩 한 줄** (6.3) — 「U0001 이 KODEX 레버리지를 담을까?」가 1 이 6개인 한 줄이 됩니다

**어떻게 계산하나**
1. 입력 한 줄에서 **1 인 열**을 찾습니다 — 6개
2. 그 열마다 붙어 있는 **잠재 요인**(숫자 8개)을 꺼냅니다
3. 6개를 **둘씩 짝짓습니다** — 6 × 5 ÷ 2 = 15짝
4. 짝마다 두 잠재 요인의 **내적**을 구합니다
5. 15개 내적을 **모두 더합니다.** 여기에 열마다 붙은 숫자 하나(가중치)도 더합니다
6. 더한 값을 **0 \~ 1 사이의 확률**로 바꿉니다
   - 이 변환을 **시그모이드(sigmoid)** 라고 합니다
   - 큰 양수는 1 에 가깝게, 큰 음수는 0 에 가깝게, 0 은 0.5 가 됩니다

---

#### 자세히

**1. 4주차 행렬분해를 다시 보면**
- 4주차 점수 = **사용자 요인 · 아이템 요인** (같은 자리끼리 곱해 더한 값, 내적)
- 사용자 U0001 과 종목 KODEX 레버리지 **한 짝**만 봅니다.
- 그래서 「KODEX 레버리지의 위험도가 5 다」 같은 상품 정보를 **넣을 자리가 없습니다.**

**2. FM 은 입력을 한 줄의 0/1 로 받습니다**
- 「U0001 이 KODEX 레버리지를 담을까?」를 원핫 인코딩으로 쓰면, 한 줄에 1 이 **6개** 들어갑니다.

| 무엇 | 1 이 들어간 열 |
|---|---|
| 사용자 | `user_id_U0001` |
| 종목 | `item_id_I079` (KODEX 레버리지) |
| 섹터 | `sector_레버리지` |
| 자산군 | `asset_class_ETF` |
| 위험도 | `risk_level_5` |
| 테마 | `theme_레버리지` |

- 나머지 열 420개는 모두 0 입니다. 이 표는 6.5 에서 실제로 만들어 봅니다.

**3. FM 의 점수 계산 — 둘씩 짝지어 내적을 더한다**
- 1 이 들어간 열마다 **잠재 요인**(숫자 몇 개)이 하나씩 붙어 있습니다. 4주차의 사용자 요인 · 아이템 요인과 같은 것입니다.
- 1 이 들어간 열을 **둘씩 짝짓고**, 짝마다 두 잠재 요인의 **내적**을 구해 **모두 더합니다.**
  - 열이 6개면 짝은 6 × 5 ÷ 2 = **15개**입니다.
  - 예 — (U0001, KODEX 레버리지) · (U0001, 위험도 5) · (KODEX 레버리지, 위험도 5) · …
- 실제 FM 에는 열마다 붙은 숫자 하나(가중치)를 더하는 부분도 있습니다. 오늘 이해할 핵심은 **짝의 내적**입니다.

**4. 그래서 무엇이 좋은가**
- (U0001, 위험도 5) 짝 덕분에 FM 은 **「U0001 은 위험도 5 종목을 잘 안 담는다」** 를 배울 수 있습니다.
- 이것은 거래가 적은 종목에도 쓰입니다. 그 종목을 담은 사람이 적어도, 위험도 · 섹터 열은 다른 종목과 함께 쓰기 때문입니다.
- 사용자와 종목 두 열만 1 이면 짝은 (사용자, 종목) 하나뿐입니다. 이때 FM 은 **4주차 행렬분해와 같은 계산**이 됩니다. 그래서 FM 을 「행렬분해를 넓힌 모델」이라고 부릅니다.

**5. 아래 코드가 하는 일**
- 열 세 개(U0001 · KODEX 레버리지 · 위험도 5)에 잠재 요인 2개씩을 **사람이 정한 숫자로** 붙입니다. 실제로는 이 숫자를 학습으로 찾습니다.
- 짝 3개의 내적을 하나씩 구하고, 더해서 점수를 냅니다.

#### 📖 FM 기초 지식 — 처음 배우는 사람을 위해

**1. 「모델」이란**
- 입력을 받아 **숫자 하나(점수 · 확률)** 를 내는 **계산 규칙**입니다.
  - 계산 규칙 안에는 **비어 있는 숫자**들이 있습니다. 이 숫자를 데이터로 채우는 일이 **학습**입니다.
- 4주차 행렬분해도 모델입니다.
  - 계산 규칙은 「사용자 요인 · 아이템 요인을 내적한다」입니다.
  - 비어 있는 숫자는 요인 안의 숫자들이고, TruncatedSVD 가 채웠습니다.
- FM 도 같습니다.
  - 계산 규칙은 「1 인 열들을 둘씩 짝지어 내적해 더한다」입니다.
  - 비어 있는 숫자는 **열마다 붙은 잠재 요인 8개와 가중치 1개**입니다.

**2. 벡터와 잠재 요인**
- **벡터(vector)** = 숫자 여러 개를 한 줄로 늘어놓은 것입니다.
  - 예 — `[0.8, 0.1]` 은 숫자 2개짜리 벡터
- **잠재 요인**은 벡터입니다. FM 에서는 열 하나마다 숫자 8개짜리 벡터가 하나씩 붙습니다.
- 8개 숫자 하나하나에 사람이 정한 이름은 없습니다.
  - 학습이 끝나면 그중 한 칸이 「위험을 좋아하는 정도」처럼 움직일 수도 있습니다.
  - 무엇을 뜻할지는 **데이터가 정합니다.**

**3. 내적이 「잘 어울리는 정도」가 되는 이유**
- 내적은 같은 자리끼리 곱해서 더합니다. 그래서 **같은 자리의 부호가 같으면 커지고, 다르면 작아집니다.**

| 두 벡터 | 내적 | 뜻 |
|---|---|---|
| `[1, 1]` 와 `[1, 1]` | 1 + 1 = **2** | 같은 방향 → 잘 어울린다 |
| `[1, 1]` 와 `[1, -1]` | 1 − 1 = **0** | 상관없다 |
| `[1, 1]` 와 `[-1, -1]` | −1 − 1 = **−2** | 반대 방향 → 안 어울린다 |

**4. 왜 「짝」이어야 하나 — 이 절의 핵심**
- `위험도 5` 열에 가중치 **하나만** 붙어 있다고 해 봅시다.
  - 가중치가 양수면 위험도 5 종목은 **모든 사람에게** 점수가 올라갑니다.
  - 음수면 **모든 사람에게** 내려갑니다.
- 실제로는 그렇지 않습니다.
  - 공격형 투자자에게 위험도 5 는 **좋은** 정보입니다.
  - 안정형 투자자에게는 **나쁜** 정보입니다.
- 「누구에게 좋은가」는 **열 하나로는 표현할 수 없고, 두 열의 짝(사용자 × 위험도 5)으로만** 표현됩니다.
  - 이렇게 두 정보가 함께 있을 때 생기는 효과를 **상호작용(interaction)** 이라고 합니다.
  - FM 은 모든 짝의 상호작용을 잠재 요인의 내적으로 계산합니다.
- 반대로 열마다 붙은 **가중치 하나**는 「누구에게나 같은 효과」를 맡습니다.
  - 예 — 많은 사람이 담는 종목은 이 가중치가 커집니다. 1주차의 인기 추천과 같은 역할입니다.

**5. 점수를 확률로 바꾸기 — 시그모이드**
- 짝의 내적을 더한 점수는 −3 도, 5 도 될 수 있습니다. 「담을 확률」로 쓰려면 0 \~ 1 사이여야 합니다.
- **시그모이드(sigmoid)** 는 어떤 숫자든 0 \~ 1 사이로 바꿉니다. 점수가 클수록 1 에 가까워지고, 순서는 바뀌지 않습니다.

| 점수 | −4 | −2 | −0.39 (6.4 의 예) | 0 | 2 | 4 |
|---|---|---|---|---|---|---|
| 확률 | 0.018 | 0.119 | 0.404 | 0.5 | 0.881 | 0.982 |

**6. FM 과 4주차 행렬분해 비교**

| | 4주차 행렬분해 | FM |
|---|---|---|
| 입력 | 사용자 번호 · 종목 번호 | 1 이 6개인 0/1 한 줄 (상품 정보 포함) |
| 잠재 요인이 붙는 곳 | 사용자 · 종목 | **모든 열** (426개) |
| 점수 계산 | 사용자 × 종목 **한 짝**의 내적 | **15짝**의 내적 합 + 열마다 가중치 |
| 숫자를 찾는 방법 | 계산 한 번 (TruncatedSVD) | 조금씩 고치기를 반복 (6.5) |
| 딥러닝인가 | 아니다 | **아니다** — 층이 없다. 「딥러닝 추천」의 바로 앞 단계 |

In [12]:
# 이 셀에서 하는 일 — 열 세 개로 FM 의 점수를 손으로 계산해 본다
# 왜 하나 — 「둘씩 짝지어 내적을 더한다」가 무슨 뜻인지 숫자로 본다. 숫자는 설명을 위해 사람이 정한 값이다

U0001_요인 = np.array([0.8, 0.1])           # U0001 열의 잠재 요인 2개
레버리지_요인 = np.array([-0.5, 0.9])       # KODEX 레버리지 열의 잠재 요인 2개
위험도5_요인 = np.array([-0.6, 0.1])        # 위험도 5 열의 잠재 요인 2개

짝1 = U0001_요인 @ 레버리지_요인             # @ = 내적. 같은 자리끼리 곱해 더한다
짝2 = U0001_요인 @ 위험도5_요인
짝3 = 레버리지_요인 @ 위험도5_요인

print(f"짝 1 (U0001, KODEX 레버리지) : 0.8×(-0.5) + 0.1×0.9    = {짝1:+.2f}   ← 4주차 행렬분해는 이것만 본다")
print(f"짝 2 (U0001, 위험도 5)       : 0.8×(-0.6) + 0.1×0.1    = {짝2:+.2f}")
print(f"짝 3 (KODEX 레버리지, 위험도 5): (-0.5)×(-0.6) + 0.9×0.1 = {짝3:+.2f}")
print(f"FM 점수 = 세 짝의 합 = {짝1 + 짝2 + 짝3:+.2f}")

짝 1 (U0001, KODEX 레버리지) : 0.8×(-0.5) + 0.1×0.9    = -0.31   ← 4주차 행렬분해는 이것만 본다
짝 2 (U0001, 위험도 5)       : 0.8×(-0.6) + 0.1×0.1    = -0.47
짝 3 (KODEX 레버리지, 위험도 5): (-0.5)×(-0.6) + 0.9×0.1 = +0.39
FM 점수 = 세 짝의 합 = -0.39


**결과 읽는 법**

- **짝 1** 은 4주차 행렬분해의 점수와 같은 계산입니다. 사용자와 종목만 봅니다.
- **짝 2** 는 「U0001 과 위험도 5」의 짝입니다. 음수(−0.47)라서 점수를 **끌어내립니다.**
  - 이런 숫자가 학습되었다면 「U0001 은 위험도 5 종목을 잘 안 담는다」는 뜻입니다.
- **짝 3** 은 사용자가 없는 짝, 「KODEX 레버리지와 위험도 5」입니다. 양수(+0.39)입니다.
  - 상품 정보끼리, 종목과 상품 정보끼리의 짝도 똑같이 계산해 더합니다.
- **마지막 줄**이 FM 의 점수입니다. 세 짝을 더해 **−0.39** 입니다.
  - 짝 1 하나만 봤을 때(−0.31)보다 더 낮습니다. 위험도 5 라는 정보가 점수를 더 끌어내렸습니다.
- 실제 FM 은 1 인 열이 6개라서 짝 15개를 더하고, 잠재 요인도 2개가 아니라 8개입니다. 계산 방법은 같습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.5 FM 학습 — 학습 데이터를 만들고, FM 을 학습해 채점한다</mark>

**6.5 에서 하는 일** — 세 단계로 FM 을 실제로 학습합니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 6.4 에서는 잠재 요인 숫자를 **사람이 정했습니다.** 실제로는 열 426개 × 8개 = 3,408개의 숫자를 정해야 합니다.
- 이 숫자들을 거래 기록에서 **자동으로 찾는 과정**이 학습(training)입니다.

**왜 중요한가**
- 오늘 배우는 학습 방법은 FM 만의 방법이 아닙니다. **MLP · DeepFM · 대부분의 딥러닝 모델이 같은 방식으로 학습합니다.**
  - 「예측한다 → 정답과 비교한다 → 차이가 줄어드는 쪽으로 숫자를 조금 고친다 → 반복한다」
- 한 번 이해하면 6.6 의 MLP 는 「계산 모양만 다른 모델」로 읽힙니다.

**알아 둘 배경 지식**
- **지도 학습 (supervised learning)** — **정답이 붙은 예시**를 보고 배우는 방식
  - 오늘의 예시 = (투자자, 종목) 짝, 정답 = 담았으면 1, 안 담았으면 0
- **이진 분류 (binary classification)** — 답이 둘 중 하나(1 또는 0)인 문제
- **4주차와의 차이**
  - 4주차 TruncatedSVD 는 **계산 한 번**으로 잠재 요인을 구했습니다
  - FM 은 그런 한 번의 계산이 없어서, **조금씩 고치기를 반복해서** 찾습니다

**어떻게 계산하나 — 학습의 순서**
1. 학습 데이터를 만든다 — 담은 짝(1)과 네거티브 샘플링으로 고른 담지 않은 짝(0)
2. 잠재 요인을 **아주 작은 무작위 숫자**로 채워 시작한다
3. 학습 데이터를 256줄씩 나눠, 줄마다 6.4 의 방법으로 **확률**을 예측한다
4. **차이 = 예측 확률 − 정답**을 구한다
   - 정답이 1 인데 0.3 으로 예측했으면 차이는 −0.7 → 이 줄의 점수를 **올리는 쪽으로** 고친다
   - 정답이 0 인데 0.9 로 예측했으면 차이는 +0.9 → 점수를 **내리는 쪽으로** 고친다
5. 그 줄에서 1 인 열들의 잠재 요인 · 가중치를 **조금** 고친다. 차이가 클수록 많이 고친다
6. 27,750줄을 다 보면 **에폭 1** 이 끝난다. 이것을 80번 반복한다
7. 학습이 끝나면 285명 × 100종목 = 28,500개 짝 모두에 확률을 매기고, 사람마다 높은 순으로 10개를 추천한다
- 4\~5 의 「차이가 줄어드는 쪽으로 조금씩 고치기」를 **경사 하강법(gradient descent)** 이라고 부릅니다

---

#### 자세히

| 단계 | 하는 일 | 쓰는 도구 |
|---|---|---|
| ① | 학습 데이터 만들기 — 담은 짝은 1, 담지 않은 짝은 0 | `recsys.training_rows` |
| ② | 학습 데이터를 원핫 인코딩 — 한 줄에 1 이 6개 | `recsys.one_hot` |
| ③ | FM 학습 → 285명 × 100종목에 점수 → 채점 | `recsys.FM` |

**1. 왜 「담지 않은 짝」이 필요한가 — 네거티브 샘플링**
- 모델은 **「이 짝이면 1, 저 짝이면 0」을 구별하는 법**을 배웁니다.
- 거래 기록에는 「담았다」만 있습니다. 1 만 있는 데이터로 배우면 모델은 **모든 짝에 1 이라고 답하면 다 맞는다**고 배웁니다. 아무것도 구별하지 못합니다.
- 그래서 사람마다 **담지 않은 종목을 무작위로 골라 0 으로 넣습니다.** 이것이 **네거티브 샘플링(negative sampling)** 입니다.
  - 「네거티브(negative)」 = 0 인 예시, 「샘플링(sampling)」 = 무작위로 고른다
  - 오늘은 사람마다 **담은 개수의 4배**를 0 으로 넣습니다.
- 「담지 않았다」가 「싫어한다」는 뜻은 아닙니다. 몰라서 안 담았을 수도 있습니다. 그래도 담은 종목보다는 덜 좋아할 가능성이 크다고 보고 0 으로 씁니다.

**2. 무작위인데 왜 매번 같은 결과가 나오나 — 씨앗(seed)**
- 무작위로 고르면 돌릴 때마다 다른 종목이 뽑히고 점수도 바뀝니다.
- `seed=42` 처럼 **씨앗(random seed)** 을 고정하면 몇 번을 돌려도 같은 종목이 뽑힙니다. 4주차 TruncatedSVD 의 `random_state=42` 와 같은 뜻입니다.
- 그래서 여러분 화면의 숫자가 이 노트북의 숫자와 같습니다.

**3. 이 도구들을 왜 `recsys.py` 에 넣었나**
- 오늘의 목표는 FM 과 MLP 가 **무엇을 하는지 이해하고 결과를 비교하는 것**입니다.
- 학습 데이터를 만드는 반복문과 FM 의 학습 계산은 길어서 `recsys.py` 로 옮겼습니다. 궁금하면 `recsys.py` 의 맨 아래를 열어 보세요.
- FM 은 scikit-learn 에 들어 있지 않아서 **numpy 로 직접 짰습니다.** 그래서 따로 설치할 것이 없습니다.

#### 📖 학습 기초 지식 — FM 과 MLP 가 똑같이 쓰는 말

**1. 손실 (loss) — 「얼마나 틀렸나」를 숫자 하나로**
- 학습 데이터 줄마다 「예측 확률」과 「정답」의 차이가 있습니다. 이것을 모두 모아 **숫자 하나**로 만든 것이 손실입니다.
  - 정답 1 인데 0.9 로 예측 → 조금 틀림 → 손실 작음
  - 정답 1 인데 0.1 로 예측 → 많이 틀림 → 손실 큼
- **학습 = 손실을 줄이는 쪽으로 숫자를 고치는 일**입니다.

**2. 경사 하강법 (gradient descent) — 어느 쪽으로 고칠지 정하는 방법**
- 숫자 하나를 아주 조금 바꿨을 때 손실이 늘어나는지 줄어드는지를 계산합니다. 이 「늘어나는 방향과 크기」를 **기울기(gradient)** 라고 합니다.
- 손실이 **줄어드는 방향으로** 숫자를 조금 옮깁니다. 이것을 수천 번 반복하면 손실이 점점 작아집니다.

**3. 학습률 (learning rate) — 한 번에 얼마나 고칠지**
- `recsys.FM` 의 `lr=0.01` 이 학습률입니다.
  - 너무 크면 숫자가 지나치게 움직여 손실이 오히려 커집니다(실제로 이 노트북을 만들 때 0.05 로는 숫자가 발산했습니다).
  - 너무 작으면 80 에폭을 돌아도 거의 배우지 못합니다.

**4. 미니배치 (mini-batch) — 몇 줄씩 보고 고칠지**
- 27,750줄을 한 줄 볼 때마다 고치면 느리고, 다 본 뒤 한 번 고치면 너무 드뭅니다.
- 그래서 **256줄씩** 묶어 보고 한 번 고칩니다. 이 묶음이 미니배치입니다.
  - 에폭 1 = 미니배치 약 109번 (27,750 ÷ 256)
  - 에폭 80 = 약 8,700번 고친다

**5. 왜 매번 같은 결과가 나오나 — 씨앗(seed)**
- 학습은 **무작위**로 시작합니다. 처음 잠재 요인, 0 으로 쓸 종목, 줄을 보는 순서가 모두 무작위입니다.
- `seed=42` 로 무작위의 시작점을 고정해서, 누가 돌려도 같은 결과가 나오게 했습니다.

#### 📖 학습 더 자세히 — 숫자 하나가 고쳐지는 과정

**1. 손실을 실제로 어떻게 재나 — 로그 손실 (log loss)**
- 오늘 FM 과 MLP 는 **로그 손실**을 씁니다. 「정답에 준 확률」이 낮을수록 손실이 **급격히** 커지는 계산입니다.

| 정답이 1 일 때 모델이 준 확률 | 0.9 | 0.5 | 0.1 | 0.01 |
|---|---|---|---|---|
| 그 줄의 손실 | 0.105 | 0.693 | 2.303 | 4.605 |

- 정답이 0 이면 반대로, 「0 일 확률(= 1 − 예측 확률)」로 같은 표를 읽습니다.
- 확신하고 틀린 줄(정답 1 인데 0.01)이 손실을 가장 크게 만듭니다. 그래서 학습은 **크게 틀린 줄부터** 고치게 됩니다.
- 학습 데이터 전체의 손실 = 모든 줄의 손실을 **평균**한 값입니다.
- 아무것도 모르고 모든 줄에 0.5 라고 답하면 손실은 0.693 입니다. 학습이 되고 있다면 손실이 이 값보다 **작아져야** 합니다.

**2. 한 번 고치기 — 순서대로**
1. 한 줄을 넣고 확률을 예측한다 — 예: 0.27
2. 정답과 비교한다 — 정답 1, 차이 = 0.27 − 1 = **−0.73**
3. 차이가 음수 → 「확률을 올려야 한다」 → 점수를 올리는 쪽으로 숫자를 옮긴다
4. 얼마나 옮기나 = 차이 × 학습률
   - 차이가 클수록, 학습률이 클수록 많이 옮긴다
5. 다음 줄로 넘어간다
- 바로 아래 셀에서 **점수 하나만** 있는 가장 작은 모델로 이 과정을 다섯 번 따라가 봅니다.

**3. 학습이 끝났는지 어떻게 아나**
- 에폭마다 손실을 기록해서 **더 이상 거의 줄지 않으면** 충분히 배운 것으로 봅니다.
- 6.5 · 6.6 에서 학습이 끝난 뒤 이 기록(`loss_curve_`)을 직접 봅니다.

**4. 덜 배움과 너무 배움**

| 상태 | 학습 데이터 손실 | 새 데이터 성적 | 원인 |
|---|---|---|---|
| **과소적합 (underfitting)** | 큼 | 나쁨 | 덜 배웠다 — 에폭이 너무 적거나, 학습률이 너무 작거나, 모델이 너무 단순하다 |
| 알맞음 | 적당히 작음 | 좋음 | |
| **과적합 (overfitting)** | **아주 작음** | 나쁨 | 너무 배웠다 — 학습 데이터를 외웠다. 모델이 데이터에 비해 너무 복잡하다 |

- 학습 데이터 손실만 보면 과적합을 알 수 없습니다. 손실이 작을수록 좋아 보이기 때문입니다.
- 그래서 **학습에 쓰지 않은 데이터**(우리는 다음 달 기록)로 반드시 따로 채점합니다.

**5. `fit` 과 `predict` — 학습과 예측은 다른 단계**
- `fit(X, y)` — **학습**. 정답(`y`)을 보며 숫자를 고친다. 시간이 걸린다
- `predict(X)` / `predict_proba(X)` — **예측**. 숫자를 고치지 않고 계산만 한다. 빠르다
- scikit-learn 의 모든 모델이 이 두 이름을 씁니다. `recsys.FM` 도 같은 이름으로 만들었습니다.
- 7주차에 배우는 「학습과 서빙의 분리」가 바로 이 두 단계를 따로 돌리는 것입니다.

In [13]:
# 이 셀에서 하는 일 — 점수 하나짜리 가장 작은 모델을 다섯 번 고쳐 본다
# 왜 하나 — 「예측 → 정답과 비교 → 조금 고치기」가 반복되면 손실이 줄어드는 것을 숫자로 본다

점수 = -1.0          # 처음 점수. 아무렇게나 정한 값이다
정답 = 1             # 이 줄의 정답 — 담았다
학습률 = 1.0         # 한 번에 얼마나 고칠지. 눈에 잘 보이게 크게 잡았다

for 번째 in range(1, 6):                              # 다섯 번 고친다
    확률 = 1 / (1 + np.exp(-점수))                      # 시그모이드 — 점수를 확률로
    손실 = -np.log(확률)                                # 로그 손실 (정답이 1 일 때)
    차이 = 확률 - 정답                                  # 예측 − 정답
    print(f"{번째}번째 · 점수 {점수:+.2f} → 확률 {확률:.3f} · 손실 {손실:.3f} · 차이 {차이:+.3f}")
    점수 = 점수 - 학습률 * 차이                         # 차이의 반대 방향으로 옮긴다

1번째 · 점수 -1.00 → 확률 0.269 · 손실 1.313 · 차이 -0.731
2번째 · 점수 -0.27 → 확률 0.433 · 손실 0.837 · 차이 -0.567
3번째 · 점수 +0.30 → 확률 0.574 · 손실 0.555 · 차이 -0.426
4번째 · 점수 +0.72 → 확률 0.673 · 손실 0.395 · 차이 -0.327
5번째 · 점수 +1.05 → 확률 0.741 · 손실 0.300 · 차이 -0.259


**결과 읽는 법**

- 한 줄이 「한 번 예측하고 고치기」입니다.
- **1번째** — 점수 −1 이라 확률이 0.269 입니다. 정답(1)과의 차이가 −0.731 로 큽니다.
  - 차이가 음수라서 다음 점수는 −1 − 1.0 × (−0.731) = **−0.27** 로 올라갑니다.
- 2번째부터 점수 · 확률이 올라가고 **손실이 매번 줄어듭니다** (1.313 → … → 0.300).
- 차이가 줄어들수록 **고치는 양도 줄어듭니다.** 정답에 가까워질수록 천천히 움직입니다.
- FM 과 MLP 는 이 일을 숫자 수천 · 수만 개에 동시에, 미니배치마다 반복합니다.

In [14]:
# 이 셀에서 하는 일 — ① 학습 데이터를 만든다. 담은 짝은 1, 담지 않은 짝은 0
# 왜 하나 — 0 이 있어야 모델이 「담을 짝」과 「안 담을 짝」을 구별하는 법을 배운다

학습_데이터 = recsys.training_rows(train, items, neg=4, seed=42)   # neg=4 = 담은 개수의 4배를 0 으로

print(f"학습 데이터 {len(학습_데이터):,}줄")
print(학습_데이터["label"].value_counts())       # label 열에서 1 과 0 이 각각 몇 줄인지 센다
학습_데이터.head(3)                               # 맨 위 세 줄

학습 데이터 27,750줄
label
0    20655
1     7095
Name: count, dtype: int64


,user_id,item_id,label
0,U0001,I006,1
1,U0001,I007,1
2,U0001,I013,1


**결과 읽는 법**

- 학습 데이터는 모두 **27,750줄**입니다. 한 줄이 「사용자 한 명 × 종목 하나」 짝입니다.
- `label` 은 정답 열입니다.
  - **1 이 7,095줄** — 학습 구간에서 실제로 담은 짝입니다. 같은 종목을 여러 번 산 기록은 한 줄로 셉니다.
  - **0 이 20,655줄** — 네거티브 샘플링으로 넣은 「담지 않은 짝」입니다.
  - 0 이 1 의 4배보다 조금 적습니다. 종목을 많이 담은 사람은 남은 종목이 적어서 4배를 다 채우지 못하기 때문입니다.
- 아래 표는 맨 위 세 줄입니다. 모두 U0001 이 담은 종목이라 `label` 이 1 입니다.

In [15]:
# 이 셀에서 하는 일 — ② 학습 데이터를 원핫 인코딩하고, 첫 줄에서 1 인 열을 본다
# 왜 하나 — FM 과 MLP 는 숫자만 받는다. 6.3 의 원핫 인코딩을 사용자 · 종목 번호에도 쓴다

X = recsys.one_hot(학습_데이터, items)   # 사용자 · 종목 · 상품 정보 네 가지를 0/1 열로 펼친다
y = 학습_데이터["label"]                 # 정답 — 1(담음) 또는 0(안 담음)

print(f"X {X.shape}   ← 학습 데이터 27,750줄 × 0/1 열")

첫줄_1인_열 = []                          # 첫 줄에서 1 인 열 이름을 담을 빈 목록
for 열 in X.columns:                      # 열을 하나씩 본다
    if X.iloc[0][열] == 1:                # iloc[0] = 맨 첫 줄. 1 인 열만 모은다
        첫줄_1인_열.append(열)
print(f"첫 줄 (U0001, I006 {종목이름['I006']}) → 1 인 열 {len(첫줄_1인_열)}개 :")
print(첫줄_1인_열)

X (27750, 426)   ← 학습 데이터 27,750줄 × 0/1 열
첫 줄 (U0001, I006 TIGER 리츠부동산인프라) → 1 인 열 6개 :
['user_id_U0001', 'item_id_I006', 'sector_리츠', 'asset_class_ETF', 'risk_level_2', 'theme_배당']


**결과 읽는 법**

- `X (27750, 426)` — 학습 데이터 27,750줄이 **0/1 열 426개**로 펼쳐졌습니다.
  - 사용자 285 + 종목 100 + 상품 정보 41 = **426**
  - 상품 정보 41개는 6.3 에서 만든 아이템 피처 행렬의 열과 같습니다.
- 첫 줄에서 1 인 열은 **6개**입니다 — 사용자 1 · 종목 1 · 섹터 1 · 자산군 1 · 위험도 1 · 테마 1.
  - 6.4 의 표와 같은 모양입니다. FM 은 이 6개를 둘씩 짝지어 15개 짝의 내적을 더합니다.
- `X` 는 입력, `y` 는 정답입니다. 머신러닝에서 입력을 `X`, 정답을 `y` 로 부르는 것이 관례입니다.

**③ FM 학습 — 아래 셀에서 일어나는 일**

1. **`recsys.FM(k=8, epochs=80)`** — FM 모델을 하나 만듭니다.
   - `k=8` — 열마다 붙는 잠재 요인의 개수. 4주차의 `n_components=8` 과 같은 뜻입니다.
   - `epochs=80` — **에폭(epoch)** 80. 학습 데이터 27,750줄을 처음부터 끝까지 **80번** 봅니다.
2. **`.fit(X, y)`** — 학습합니다.
   - 처음에는 잠재 요인이 아무렇게나 정해져 있습니다.
   - 한 줄씩 「예측한 확률」과 「정답(1 또는 0)」의 차이를 보고, 차이가 줄어드는 쪽으로 잠재 요인을 **조금씩** 고칩니다.
   - 이것을 80번 반복합니다. 한 번에 많이 고치면 숫자가 튀어서 조금씩 여러 번 고칩니다.
3. **`.predict(...)`** — 짝마다 **담을 확률(0 \~ 1)** 을 냅니다.
   - 285명 × 100종목 = **28,500개 짝** 모두에 확률을 매깁니다.
   - 확률이 높은 종목부터 10개를 추천합니다. 6.1 의 4주차 추천과 같은 방식입니다.
4. **두 번 학습합니다.**
   - 상품 정보를 **넣은** FM 과 **넣지 않은** FM(사용자 · 종목 열만) — 상품 정보가 도움이 됐는지 비교합니다.

**학습 시간** — 한 번에 10 \~ 15초쯤 걸립니다. 셀 왼쪽이 돌아가는 동안 기다려 주세요.

In [16]:
# 이 셀에서 하는 일 — 285 × 100 점수 행렬을 받아 채점하는 함수를 만든다
# 왜 하나 — 오늘 모델 네 개(행렬분해 · FM · MLP 1층 · MLP 2층)를 모두 같은 방식으로 채점한다

전체_짝 = recsys.all_pairs(사용자_목록, items)   # 점수를 매길 모든 짝 — 285명 × 100종목 = 28,500줄

def 점수_행렬로_채점(점수_행렬):
    """285 × 100 점수 행렬로 6.1 과 같은 방식의 추천을 만들고 Recall@10 을 잰다."""
    def 추천(사람, 이미):                                  # 함수 안에서 추천 함수를 하나 만든다
        if 사람 not in 사용자_목록:                         # 신규 투자자는 6.1 과 같이 인기 목록
            return recsys.take(전체_인기순위, 이미)
        점수 = 점수_행렬[사용자_목록.index(사람)]            # 이 사람의 종목 100개 점수
        return recsys.take(점수순으로_종목_정렬(점수), 이미)
    점수, _ = recsys.recall_at_k(추천, train, test)
    return 점수

print(f"행렬분해(4주차) 다시 채점 — Recall@10 = {점수_행렬로_채점(사용자_요인 @ 아이템_요인):.4f}")

행렬분해(4주차) 다시 채점 — Recall@10 = 0.3918


**결과 읽는 법**

- 6.1 의 4주차 추천을 새 함수로 다시 채점한 값입니다.
  - `사용자_요인 @ 아이템_요인` 은 285 × 100 점수 행렬입니다. 285명 각각의 종목 100개 점수입니다.
- **0.3918** 로 6.1 과 같습니다. 새 함수가 6.1 과 똑같이 채점한다는 확인입니다.
- 이제 FM 과 MLP 도 285 × 100 점수 행렬만 만들면 이 함수로 채점할 수 있습니다.

In [17]:
# 이 셀에서 하는 일 — ③ FM 을 두 번 학습해 채점한다. 상품 정보를 넣은 것과 넣지 않은 것
# 왜 하나 — FM 이 4주차보다 나은지, 상품 정보가 점수를 올렸는지 본다

import time                                         # 학습 시간을 재는 도구

시작 = time.time()
FM_모델 = recsys.FM(k=8, epochs=80).fit(X, y)                      # 상품 정보를 넣은 FM
FM_점수 = FM_모델.predict(recsys.one_hot(전체_짝, items))          # 28,500개 짝의 담을 확률
FM_점수 = FM_점수.reshape(len(사용자_목록), len(아이템_목록))       # reshape = 285 × 100 행렬로 다시 접는다
FM_결과 = 점수_행렬로_채점(FM_점수)
print(f"FM (상품 정보 넣음)   — Recall@10 = {FM_결과:.4f} · {time.time() - 시작:.0f}초")

시작 = time.time()
X_정보없음 = recsys.one_hot(학습_데이터, items, features=False)   # 사용자 · 종목 열만 — 상품 정보 없음
FM_모델_정보없음 = recsys.FM(k=8, epochs=80).fit(X_정보없음, y)
FM_점수_정보없음 = FM_모델_정보없음.predict(recsys.one_hot(전체_짝, items, features=False))
FM_점수_정보없음 = FM_점수_정보없음.reshape(len(사용자_목록), len(아이템_목록))
FM_결과_정보없음 = 점수_행렬로_채점(FM_점수_정보없음)
print(f"FM (상품 정보 없음)   — Recall@10 = {FM_결과_정보없음:.4f} · {time.time() - 시작:.0f}초")
print(f"행렬분해(4주차)       — Recall@10 = {지난주_점수:.4f}")

FM (상품 정보 넣음)   — Recall@10 = 0.3463 · 12초


FM (상품 정보 없음)   — Recall@10 = 0.3670 · 11초
행렬분해(4주차)       — Recall@10 = 0.3918


**결과 읽는 법**

- 세 줄 모두 같은 학습 구간으로 배우고, 같은 다음 달 기록으로 채점한 점수입니다.
- **FM (상품 정보 넣음)** — **0.3463** 입니다.
  - 4주차 행렬분해(0.3918)보다 **낮습니다.**
- **FM (상품 정보 없음)** — **0.3670** 입니다.
  - 상품 정보를 넣은 FM 보다 조금 높습니다. 사용자 · 종목 열만 쓰므로 4주차와 같은 재료를 썼습니다.
- **행렬분해(4주차)** — **0.3918** 로 셋 중 가장 높습니다.
- 끝의 초는 학습과 채점에 걸린 시간입니다. 4주차 TruncatedSVD 는 1초도 걸리지 않았습니다.
- ⚠ Colab 에서는 컴퓨터가 달라서 점수의 마지막 자리가 조금 다를 수 있습니다.

In [18]:
# 이 셀에서 하는 일 — 방금 학습한 FM 의 에폭별 손실을 본다 (다시 학습하지 않는다)
# 왜 하나 — 학습이 실제로 손실을 줄였는지, 언제부터 거의 줄지 않는지 확인한다

for 에폭 in [1, 5, 10, 20, 40, 60, 80]:
    print(f"에폭 {에폭:2d} · 학습 데이터 손실 {FM_모델.loss_curve_[에폭 - 1]:.4f}")

에폭  1 · 학습 데이터 손실 0.5477
에폭  5 · 학습 데이터 손실 0.4961
에폭 10 · 학습 데이터 손실 0.4790
에폭 20 · 학습 데이터 손실 0.4336
에폭 40 · 학습 데이터 손실 0.3800
에폭 60 · 학습 데이터 손실 0.3603
에폭 80 · 학습 데이터 손실 0.3582


**결과 읽는 법**

- 한 줄이 「그 에폭이 끝났을 때 학습 데이터 27,750줄의 평균 손실」입니다.
- **에폭 1 → 80** 동안 손실이 **0.5477 → 0.3582** 로 줄었습니다. 학습이 실제로 일어났다는 확인입니다.
  - 모두 0.5 라고 답할 때의 손실(0.693)보다 처음부터 작습니다. 첫 에폭만에 「1 보다 0 이 많다」는 것을 배웠기 때문입니다.
- **에폭 60 → 80** 사이에는 0.3603 → 0.3582 로 **거의 줄지 않습니다.** 이 정도면 충분히 배운 것으로 봅니다.
  - 그래서 에폭을 80 으로 정했습니다. 더 돌려도 시간만 더 들고 크게 달라지지 않습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.6 MLP — 층을 거쳐 확률을 내는 신경망, 층 1개와 2개</mark>

**6.6 에서 하는 일** — 같은 학습 데이터(`X`, `y`)로 **MLP** 를 학습합니다. 층이 1개인 것과 2개인 것을 비교합니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- MLP 는 **딥러닝의 가장 기본적인 모델**입니다. 「딥러닝 추천」이라는 6주차 주제가 실제로 무엇인지 직접 확인합니다.
- FM 은 「짝끼리 내적」이라는 계산 모양을 **사람이 정해 두었습니다.** MLP 는 계산 모양을 정하지 않고 **데이터에 맞춰 찾게 합니다.** 둘 중 어느 쪽이 우리 데이터에 맞는지 봅니다.

**왜 중요한가**
- 대형 서비스(동영상 · 쇼핑 · 광고)의 추천 모델에는 대부분 MLP 층이 들어 있습니다.
- 그런데 **언제나 좋은 것은 아닙니다.** 언제 쓰면 안 되는지를 아는 것이 이 절의 목표입니다. 답은 6.7 의 숫자가 알려 줍니다.

**알아 둘 배경 지식**
- **가중치 (weight)** — 입력마다 곱하는 숫자. 그 입력이 결과에 얼마나 영향을 주는지 나타냅니다
- **가중합 (weighted sum)** — 입력마다 가중치를 곱해서 모두 더한 값
  - 예 — 입력 `[1, 0, 1]`, 가중치 `[0.5, 2.0, -0.3]` → 1×0.5 + 0×2.0 + 1×(−0.3) = 0.2
  - 원핫 입력에서는 **1 인 열의 가중치만 더해집니다**
- **학습 방법** (6.5) — FM 과 같습니다. 예측 → 정답과 비교 → 가중치를 조금 고친다 → 반복

**어떻게 계산하나 — 한 줄이 확률이 되기까지**
1. 입력 — 한 줄의 0/1 열 426개 (6.5 의 `X` 와 같은 것)
2. **뉴런 하나**가 하는 일 — 426개 입력의 **가중합**을 구하고, 음수면 0 으로 바꾼다 (ReLU)
3. **층 하나** = 뉴런 32개. 뉴런마다 가중치가 달라서 서로 다른 값 32개가 나온다
4. 층이 2개면, 첫 층이 만든 값들을 **입력으로** 두 번째 층이 같은 계산을 한 번 더 한다
5. 마지막 층의 값들을 가중합해서 하나로 모으고, 시그모이드(6.4)로 **0 \~ 1 확률**로 바꾼다
6. 학습은 6.5 와 같다 — 확률과 정답의 차이를 보고 모든 가중치를 조금씩 고치기를 반복한다

---

#### 자세히

**1. MLP 의 계산 — 층을 차례로 거친다**

| 단계 | 들어가는 것 | 하는 계산 | 나오는 것 |
|---|---|---|---|
| 입력 | 한 줄의 0/1 열 **426개** | — | 426개 |
| 층 1 | 426개 | 426개에 가중치를 곱해 더한 값을 **32개**(1층 모델) 또는 **64개**(2층 모델) 만든다. 음수는 0 으로 바꾼다 | 32개 또는 64개 |
| 층 2 (2층 모델만) | 64개 | 같은 계산으로 **32개** 를 만든다 | 32개 |
| 출력 | 마지막 층의 32개 | 곱해 더해서 하나로 모은 뒤 0 \~ 1 로 바꾼다 | **담을 확률** 1개 |

- **층(layer)** = 「곱해 더하고, 음수는 0 으로」를 한 번 하는 단계입니다.
  - 층마다 만드는 값의 개수(32 · 64)를 **뉴런(neuron)** 또는 **노드(node)** 의 개수라고 부릅니다.
  - 「음수는 0 으로」 바꾸는 계산을 **ReLU** 라고 합니다. 이 계산이 있어야 층을 쌓는 의미가 생깁니다.
- 곱하는 가중치가 모두 **학습으로 정해지는 숫자**입니다. 처음에는 아무렇게나 정해져 있고, 정답과의 차이를 보며 조금씩 고칩니다(FM 과 같은 방식).

**2. FM 과 무엇이 다른가**

| | FM | MLP |
|---|---|---|
| 계산 모양 | **정해져 있음** — 짝끼리 내적해 더한다 | **정해져 있지 않음** — 층을 거치며 데이터에 맞는 계산을 찾는다 |
| 학습할 숫자 | 열 426개 × 잠재 요인 8개 ≈ **3,800개** | 1층 모델 ≈ **1만 4천 개** · 2층 모델 ≈ **2만 9천 개** |
| 장점 | 데이터가 적어도 비교적 안정적이다 | 데이터가 많으면 사람이 생각 못 한 관계까지 찾는다 |

- 「학습할 숫자가 많다」 = 모델이 **더 복잡한 관계를 표현할 수 있다**는 뜻입니다. 그만큼 숫자를 정하는 데 **더 많은 데이터가 필요합니다.**

**3. 어떤 도구를 쓰나**
- scikit-learn 의 **`MLPClassifier`** 입니다. Colab 에 처음부터 들어 있습니다.
  - `hidden_layer_sizes=(32,)` — 층 1개, 뉴런 32개
  - `hidden_layer_sizes=(64, 32)` — 층 2개, 첫 층 64개 · 둘째 층 32개
  - `max_iter=30` — 에폭 30
  - `random_state=42` — 씨앗. 처음 가중치를 늘 같게 정합니다
- `predict_proba` 는 확률을 0 일 확률 · 1 일 확률 두 열로 줍니다. `[:, 1]` 로 **1(담음)일 확률**만 꺼냅니다.

#### 📖 MLP 기초 지식 — 처음 배우는 사람을 위해

**1. 이름 풀이**
- **MLP = Multi-Layer Perceptron** (다층 퍼셉트론)
  - **퍼셉트론(perceptron)** — 1958년에 나온 가장 단순한 인공 신경망. 「가중합을 구해 기준을 넘는지 본다」가 전부입니다
  - **다층(multi-layer)** — 이것을 여러 층으로 쌓았다는 뜻
- **신경망(neural network)** 이라는 이름은 뇌의 신경세포 연결 방식에서 따왔습니다. 실제 계산은 곱하기와 더하기입니다.

**2. 뉴런 하나가 하는 일 — 세 단계**
1. 입력마다 **가중치**를 곱해 모두 더한다 (가중합)
2. **편향(bias)** 이라는 숫자 하나를 더한다 — 가중치와 함께 학습되는 숫자
3. 결과가 음수면 0 으로 바꾼다 (**ReLU**)
- 바로 아래 셀에서 숫자로 직접 계산해 봅니다.

**3. 왜 ReLU 가 필요한가**
- ReLU 가 없으면 층이 하는 일은 「곱하고 더하기」뿐입니다.
- 「곱하고 더하기」를 두 번 하면, 처음부터 다른 숫자로 **한 번** 곱하고 더한 것과 결과가 같습니다.
  - 그러면 층을 10개 쌓아도 층 1개와 다를 것이 없습니다.
- 음수를 0 으로 꺾는 ReLU 가 끼어야, 층을 쌓을수록 **더 복잡한 관계**를 표현할 수 있습니다.

**4. 은닉층 (hidden layer)**
- 입력과 출력 **사이**에 있는 층입니다. 계산 중간 결과가 밖으로 보이지 않아서 「은닉(hidden)」이라고 부릅니다.
- `MLPClassifier` 의 `hidden_layer_sizes` 가 바로 은닉층의 크기입니다.
  - `(32,)` = 은닉층 1개, 뉴런 32개
  - `(64, 32)` = 은닉층 2개, 뉴런 64개 → 32개
- 보통 은닉층이 2개 이상이면 「깊다」고 하고, 이런 신경망을 다루는 분야를 **딥러닝**이라고 부릅니다.

**5. 학습할 숫자(파라미터)는 몇 개인가 — 직접 세기**
- 층 하나의 숫자 = (입력 개수 × 뉴런 개수) + 뉴런 개수(편향)
- 층 1개 모델 `(32,)`
  - 은닉층: 426 × 32 + 32 = 13,664
  - 출력: 32 + 1 = 33
  - 합계 **13,697개** ≈ 1만 4천
- 층 2개 모델 `(64, 32)`
  - 첫 은닉층: 426 × 64 + 64 = 27,328
  - 둘째 은닉층: 64 × 32 + 32 = 2,080
  - 출력: 32 + 1 = 33
  - 합계 **29,441개** ≈ 2만 9천
- 비교 — FM 은 426 × 8 + 426 + 1 = **3,835개**, 학습 데이터의 1 은 **7,095줄**

**6. 세 모델 비교**

| | 행렬분해 | FM | MLP |
|---|---|---|---|
| 계산 모양 | 한 짝의 내적 | 모든 짝의 내적 | 가중합 → ReLU 를 층마다 반복 |
| 계산 모양을 누가 정하나 | 사람 | 사람 | **데이터** (가중치가 정해지면서) |
| 딥러닝인가 | 아니다 | 아니다 | 층이 2개 이상이면 그렇다 |
| 점수가 왜 나왔는지 설명 | 비교적 쉽다 | 짝별로 나눠 볼 수 있다 | **어렵다** — 숫자 수만 개가 섞여 있다 |

- 마지막 줄은 금융에서 특히 중요합니다. 고객이나 감독 기관이 「왜 이 상품을 추천했나」를 물으면 답할 수 있어야 하기 때문입니다.

In [19]:
# 이 셀에서 하는 일 — 입력 3개 · 뉴런 2개 · 출력 1개짜리 아주 작은 MLP 를 손으로 계산한다
# 왜 하나 — 「가중합 → 편향 → ReLU」가 무슨 뜻인지 숫자로 본다. 숫자는 설명을 위해 사람이 정한 값이다

입력 = np.array([1, 0, 1])                  # 0/1 입력 3개 (원핫 한 줄의 일부라고 생각한다)

가중치_A = np.array([0.5, 2.0, -0.3])       # 뉴런 A 의 가중치 3개
가중치_B = np.array([-0.4, 1.0, -0.2])      # 뉴런 B 의 가중치 3개

가중합_A = 입력 @ 가중치_A + 0.1             # 가중합 + 편향 0.1
가중합_B = 입력 @ 가중치_B + 0.0             # 가중합 + 편향 0
뉴런_A = max(가중합_A, 0)                    # ReLU — 음수면 0
뉴런_B = max(가중합_B, 0)

출력_점수 = 뉴런_A * 1.5 + 뉴런_B * 0.8 - 0.2    # 출력층도 가중합 + 편향
확률 = 1 / (1 + np.exp(-출력_점수))              # 시그모이드 — 0 ~ 1 로 바꾼다

print(f"뉴런 A : 1×0.5 + 0×2.0 + 1×(-0.3) + 0.1 = {가중합_A:+.2f} → ReLU → {뉴런_A:.2f}")
print(f"뉴런 B : 1×(-0.4) + 0×1.0 + 1×(-0.2) + 0 = {가중합_B:+.2f} → ReLU → {뉴런_B:.2f}")
print(f"출력   : {뉴런_A:.2f}×1.5 + {뉴런_B:.2f}×0.8 - 0.2 = {출력_점수:+.2f} → 시그모이드 → 확률 {확률:.3f}")

뉴런 A : 1×0.5 + 0×2.0 + 1×(-0.3) + 0.1 = +0.30 → ReLU → 0.30
뉴런 B : 1×(-0.4) + 0×1.0 + 1×(-0.2) + 0 = -0.60 → ReLU → 0.00
출력   : 0.30×1.5 + 0.00×0.8 - 0.2 = +0.25 → 시그모이드 → 확률 0.562


**결과 읽는 법**

- **뉴런 A** — 가중합이 +0.30 이라 ReLU 를 지나도 **0.30** 그대로입니다.
  - 입력의 둘째 칸이 0 이라서, 가중치 2.0 은 결과에 영향을 주지 못했습니다. 원핫 입력에서는 **1 인 칸의 가중치만** 더해집니다.
- **뉴런 B** — 가중합이 −0.60 이라 ReLU 가 **0** 으로 바꿨습니다.
  - 이 입력에 대해 뉴런 B 는 「꺼져」 있습니다. 다른 입력에서는 켜질 수 있습니다.
- **출력** — 두 뉴런 값을 다시 가중합하고 시그모이드로 바꿔 **확률 0.562** 가 나왔습니다.
- 6.6 의 실제 MLP 는 입력 426개 · 뉴런 32개(또는 64개 → 32개)로 같은 계산을 합니다. 이 가중치와 편향 수만 개를 6.5 와 같은 방법으로 학습합니다.

In [20]:
# 이 셀에서 하는 일 — MLP 를 층 1개, 층 2개로 학습해 각각 채점한다
# 왜 하나 — 층을 쌓아 모델을 깊게 만들면 점수가 오르는지 본다

from sklearn.neural_network import MLPClassifier    # scikit-learn 의 MLP
import warnings
warnings.filterwarnings("ignore")                   # 「30번으로는 학습이 덜 끝났다」 경고를 숨긴다. 결과에는 영향 없다

X_전체 = recsys.one_hot(전체_짝, items)               # 점수를 매길 28,500개 짝 — 학습 데이터와 같은 426열

시작 = time.time()
MLP_1층 = MLPClassifier(hidden_layer_sizes=(32,), max_iter=30, random_state=42).fit(X, y)
MLP_1층_점수 = MLP_1층.predict_proba(X_전체)[:, 1].reshape(len(사용자_목록), len(아이템_목록))
MLP_1층_결과 = 점수_행렬로_채점(MLP_1층_점수)
print(f"MLP 층 1개 (32)      — Recall@10 = {MLP_1층_결과:.4f} · {time.time() - 시작:.0f}초")

시작 = time.time()
MLP_2층 = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=30, random_state=42).fit(X, y)
MLP_2층_점수 = MLP_2층.predict_proba(X_전체)[:, 1].reshape(len(사용자_목록), len(아이템_목록))
MLP_2층_결과 = 점수_행렬로_채점(MLP_2층_점수)
print(f"MLP 층 2개 (64, 32)  — Recall@10 = {MLP_2층_결과:.4f} · {time.time() - 시작:.0f}초")

MLP 층 1개 (32)      — Recall@10 = 0.3283 · 2초


MLP 층 2개 (64, 32)  — Recall@10 = 0.2580 · 3초


**결과 읽는 법**

- **MLP 층 1개** — **0.3283** 입니다.
  - FM (상품 정보 넣음, 0.3463)과 비슷하고, 4주차 행렬분해(0.3918)보다 낮습니다.
- **MLP 층 2개** — **0.2580** 입니다.
  - 층을 하나 더 쌓았더니 점수가 **오히려 내려갔습니다.**
- 끝의 초는 학습과 채점에 걸린 시간입니다.
- 왜 깊게 만들었는데 내려갔는지는 6.7 에서 숫자를 하나 더 보고 생각해 봅니다.

In [21]:
# 이 셀에서 하는 일 — 두 MLP 의 에폭별 손실을 본다 (다시 학습하지 않는다)
# 왜 하나 — 층 1개와 2개가 학습 데이터를 얼마나 잘 맞추게 되었는지 비교한다

for 에폭 in [1, 5, 10, 20, 30]:
    print(f"에폭 {에폭:2d} · 층 1개 손실 {MLP_1층.loss_curve_[에폭 - 1]:.4f} · 층 2개 손실 {MLP_2층.loss_curve_[에폭 - 1]:.4f}")

에폭  1 · 층 1개 손실 0.6037 · 층 2개 손실 0.5696
에폭  5 · 층 1개 손실 0.5293 · 층 2개 손실 0.4712
에폭 10 · 층 1개 손실 0.4808 · 층 2개 손실 0.3945
에폭 20 · 층 1개 손실 0.4351 · 층 2개 손실 0.2134
에폭 30 · 층 1개 손실 0.4071 · 층 2개 손실 0.0895


**결과 읽는 법**

- 한 줄이 「그 에폭이 끝났을 때의 학습 데이터 손실」입니다. 두 모델 모두 에폭마다 손실이 줄었습니다.
- **층 1개** — 0.6037 → **0.4071**. FM(에폭 80 에 0.3582)과 비슷한 범위에서 천천히 줄어듭니다.
- **층 2개** — 0.5696 → **0.0895**. 에폭 20 이후 손실이 **매우 빠르게** 작아집니다.
  - 손실 0.09 는 학습 데이터 대부분의 줄에서 정답에 아주 높은 확률을 준다는 뜻입니다.
- 학습 데이터 손실만 보면 층 2개가 훨씬 나은 모델처럼 보입니다. 그런데 바로 위 셀의 Recall@10 은 층 2개가 더 낮았습니다.
  - 이 둘이 왜 어긋나는지는 6.7 에서 다시 봅니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.7 네 모델 비교 — 학습 데이터 정답률과 Recall@10 을 함께 본다</mark>

**6.7 에서 하는 일** — 오늘 만든 모델 네 개를 표 하나에 놓고, 숫자 두 개를 함께 봅니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 모델이 네 개 생겼습니다. 이 가운데 **실제 서비스에 쓸 모델을 고르는 방법**을 배웁니다.

**왜 중요한가**
- 학습 데이터 성적만 보고 고르면 **MLP 2층**을 고르게 됩니다. 그런데 그 모델은 실제로 가장 못 맞힙니다.
- 그래서 실무에서는 반드시 **학습에 쓰지 않은 데이터**로 성적을 잽니다.
  - 3주차에 시간순으로 나눠 다음 달 기록으로 채점한 것과 같은 이유입니다.

**알아 둘 배경 지식**
- **학습 데이터 vs 평가 데이터**
  - 학습 데이터 — 모델이 보고 배운 기록. 여기서의 성적은 「이미 본 문제」의 성적
  - 평가 데이터 — 모델이 한 번도 보지 않은 기록. 우리는 다음 달 기록(`test`)
- **정답률 (accuracy)** — 맞힌 줄 ÷ 전체 줄
- **파라미터 (parameter)** — 모델이 학습으로 정하는 숫자. 이 노트북에서는 「학습할 숫자」라고 부릅니다
  - 많을수록 모델이 복잡하고, 그만큼 정하는 데 데이터가 더 많이 필요합니다

**어떻게 계산하나**
1. 모델마다 학습 데이터 27,750줄에 확률을 매기고, 0.5 보다 크면 「담음」으로 본다
2. 정답과 같은 줄의 비율 → **학습 데이터 정답률**
3. 6.5 · 6.6 에서 이미 잰 **Recall@10** (다음 달 기록) 을 옆에 놓는다
4. 두 열의 순서가 같은지 비교한다

---

#### 자세히

**1. 숫자 두 개**
- **학습 데이터 정답률** — 학습에 쓴 27,750줄에서 「담음(1) / 안 담음(0)」을 맞힌 비율
  - 확률이 0.5 보다 크면 「담음」, 아니면 「안 담음」으로 보고 정답과 비교합니다.
  - **이미 본 문제를 얼마나 잘 맞히나**입니다.
- **Recall@10** — 학습에 쓰지 않은 **다음 달 기록**을 얼마나 맞히나
  - **처음 보는 문제를 얼마나 잘 맞히나**입니다. 추천에서 진짜 중요한 숫자는 이쪽입니다.

**2. 참고할 기준**
- 학습 데이터의 74% 가 0 입니다(20,655 ÷ 27,750).
  - 그래서 모든 줄에 「안 담음」이라고만 답해도 정답률이 **0.74** 가 나옵니다.
  - 정답률은 0.74 보다 얼마나 높은지로 봅니다.
- 행렬분해(4주차)는 0/1 을 맞히는 모델이 아니어서 학습 데이터 정답률을 재지 않습니다.

In [22]:
# 이 셀에서 하는 일 — 네 모델의 학습 데이터 정답률과 Recall@10 을 표 하나로 모은다
# 왜 하나 — 「이미 본 문제」와 「처음 보는 문제」 성적을 나란히 비교한다

비교 = pd.DataFrame({
    "모델": ["행렬분해 (4주차)", "FM (상품 정보 넣음)", "MLP 층 1개", "MLP 층 2개"],
    "학습할 숫자": ["-", "약 3,800개", "약 1만 4천 개", "약 2만 9천 개"],
    "학습 데이터 정답률": ["-",
                          round(((FM_모델.predict(X) > 0.5) == y).mean(), 4),   # 0.5 보다 크면 1 로 보고 정답과 비교
                          round(MLP_1층.score(X, y), 4),                       # score = 정답률
                          round(MLP_2층.score(X, y), 4)],
    "Recall@10": [round(지난주_점수, 4), round(FM_결과, 4), round(MLP_1층_결과, 4), round(MLP_2층_결과, 4)],
})
비교

,모델,학습할 숫자,학습 데이터 정답률,Recall@10
0,행렬분해 (4주차),-,-,0.3918
1,FM (상품 정보 넣음),"약 3,800개",0.8451,0.3463
2,MLP 층 1개,약 1만 4천 개,0.8186,0.3283
3,MLP 층 2개,약 2만 9천 개,0.9785,0.2580


**결과 읽는 법**

- 한 줄이 모델 하나입니다. 위에서 아래로 갈수록 **학습할 숫자가 많은, 복잡한 모델**입니다.
- **학습 데이터 정답률** 열 — 이미 본 문제의 성적입니다.
  - FM 0.8451 · MLP 1층 0.8186 · **MLP 2층 0.9785**
  - MLP 2층은 학습 데이터 27,750줄 가운데 **98% 를 맞혔습니다.** 네 모델 중 가장 높습니다.
- **Recall@10** 열 — 처음 보는 다음 달 기록의 성적입니다.
  - **행렬분해 0.3918** 이 가장 높고, **MLP 2층 0.2580** 이 가장 낮습니다.
  - 1주차 인기 추천(0.2163)과 비교하면 MLP 2층은 인기 추천보다 조금 나은 정도입니다.
- 두 열의 순서가 **거꾸로**입니다. 이미 본 문제를 가장 잘 맞힌 모델이, 처음 보는 문제는 가장 못 맞혔습니다.

**🗣 수업에서 함께 보기**

MLP 2층은 학습 데이터를 **98%** 맞혔는데, 다음 달 기록은 네 모델 중 **가장 못 맞혔습니다**(Recall@10 0.2580).

모델 안에서 무슨 일이 일어난 걸까요? 오늘 나오는 말 열 개 중 어느 말로 설명할 수 있을까요?

- 힌트 — 학습 데이터에서 1 은 7,095줄이고, MLP 2층이 학습할 숫자는 약 2만 9천 개입니다. 두 숫자를 비교해 보세요.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」**에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.8 Wide&Deep · DeepFM — 단순한 모델과 MLP 를 한 모델로 묶은 것</mark>

> 📖 **PPT 에서 한 장으로 설명합니다.** 이 절은 복습용입니다.

**6.8 에서 하는 일** — 코드 없이 **개념만** 봅니다. 추천 모델이 어떤 순서로 발전해 왔는지 정리하고, 오늘 직접 돌리지 않는 두 모델을 소개합니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 노션 계획표의 6주차 주제가 「FM · DeepFM」입니다. 오늘 FM 과 MLP 를 직접 돌렸으니, 둘을 합친 DeepFM 을 이해할 재료가 다 모였습니다.
- 추천 · 광고 분야의 글이나 채용 공고에서 자주 나오는 이름입니다. 무엇인지 알아야 「우리에게 필요한가」를 판단할 수 있습니다.

**왜 중요한가**
- 「최신 · 복잡한 모델이 언제나 낫다」가 아니라는 것을 오늘 숫자로 확인했습니다.
- 모델을 고를 때는 모델 이름이 아니라 **우리 데이터의 크기**를 먼저 봐야 합니다.

**알아 둘 배경 지식**
- **FM** (6.4 \~ 6.5) · **MLP** (6.6) — 오늘 직접 돌린 두 모델
- **기억 (memorization)** — 자주 함께 나온 조합을 그대로 기억해서 맞히는 능력
- **일반화 (generalization)** — 본 적 없는 조합에도 맞는 답을 내는 능력
  - 6.7 의 MLP 2층은 기억은 잘했지만(정답률 0.98) 일반화는 못 했습니다(Recall 0.258)

**어떻게 계산하나 — DeepFM 의 한 줄**
1. 같은 입력 한 줄(1 인 열 6개)을 **두 부분에 동시에** 넣는다
2. FM 부분 — 6.4 처럼 짝끼리 내적해서 더한다
3. MLP 부분 — 6.6 처럼 층을 거쳐 값을 만든다
4. 두 부분의 값을 **더해서** 시그모이드로 확률을 만든다
5. 학습은 6.5 와 같다. 두 부분의 숫자를 **한 번에 같이** 고친다
- 두 부분은 열마다 붙은 잠재 요인을 **함께 씁니다.** 그래서 FM 과 MLP 를 따로 학습해 더한 것과는 다릅니다

---

#### 자세히

**1. 추천 모델의 계보**

| 순서 | 모델 | 무엇을 더했나 | 이 수업에서 |
|---|---|---|---|
| 1 | **행렬분해** (matrix factorization) | 사용자 × 종목 **한 짝**의 내적 | 4주차 · 6.1 |
| 2 | **FM** (2010) | 상품 정보까지 **모든 짝**의 내적 | 6.4 \~ 6.5 |
| 3 | **MLP** | 계산 모양을 정하지 않고 **층을 쌓아** 데이터가 찾게 한다 | 6.6 |
| 4 | **Wide&Deep** (구글, 2016) | 단순한 모델(Wide) + MLP(Deep)를 **한 모델로 묶었다** | 개념만 |
| 5 | **DeepFM** (화웨이, 2017) | Wide 자리에 **FM** 을 넣었다 → FM + MLP | 개념만 |

**2. 왜 둘을 묶었나**
- **Wide** (단순한 모델) — 「레버리지를 담은 사람은 레버리지를 또 담는다」처럼 **자주 나온 규칙을 그대로 외우는** 데 강합니다.
- **Deep** (MLP) — 「이런 사람은 저런 종목도 좋아할 것이다」처럼 **처음 보는 조합으로 넓혀 가는** 데 강합니다.
- 둘의 점수를 더해 한 번에 학습하면 두 장점을 함께 얻습니다.
- **DeepFM** 은 Wide 자리에 FM 을 넣어, 사람이 짝을 일일이 정해 주지 않아도 되게 만든 것입니다.

**3. 이 모델들이 쓰인 곳**
- Wide&Deep — 구글 플레이 스토어의 앱 추천. 사용자 10억 명 이상, 학습 데이터 5,000억 건 이상
- DeepFM — 화웨이 앱 스토어. 학습 데이터 약 10억 건(7일 동안의 클릭 기록)
- 우리 데이터는 **학습 기록 7,095건**입니다. 십만 배가 넘게 차이가 납니다.

**4. 그래서 오늘 돌리지 않습니다**
- DeepFM 은 오늘 돌린 FM 과 MLP 를 합친 것이라, 학습할 숫자가 둘보다 **더 많습니다.**
- 6.7 표에서 본 것처럼, 우리 데이터에서는 학습할 숫자가 많은 모델일수록 Recall@10 이 낮았습니다.
- 이 두 모델은 **데이터가 아주 많은 서비스를 위해 만들어진 모델**이라는 것까지 기억하면 됩니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.9 적합성 점검 — 성향에 맞지 않는 추천 세기</mark>

**6.9 에서 하는 일** — 300명에게 준 추천 3,000개 가운데 **투자 성향에 맞지 않는 것**이 몇 개인지 셉니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 6.2 에서 U0001 **한 명**에게 위험도 4 \~ 5 종목이 4개 추천된 것을 봤습니다.
- 한 명만 보고 「문제가 크다」고 할 수는 없습니다. **300명 전체**에서 몇 개인지 세어 봐야 얼마나 큰 문제인지 압니다.

**왜 중요한가**
- 숫자가 있어야 **고칠지 말지, 고친 뒤 얼마나 나아졌는지**를 판단할 수 있습니다.
  - 6.10 에서 상한을 건 뒤 이 개수가 0 이 되는지로 확인합니다.

**알아 둘 배경 지식**
- **투자성향** — 고객이 설문으로 정한 성향. 우리 데이터는 안정형 · 성장형 · 공격형
- **상품 위험도** — 우리 데이터는 1 \~ 5, 5 가 가장 위험 (실제 위험등급은 1등급이 가장 위험해서 방향이 반대)
- **적합 기준** — 성향마다 「추천해도 되는 최대 위험도」를 정한 표. 아래 2번

**어떻게 계산하나**
1. 300명을 한 명씩 본다
2. 그 사람에게 준 추천 10개를 하나씩 본다
3. 종목의 위험도가 그 사람 성향의 기준보다 **크면** 1개로 센다
4. 300명 × 10개 = 3,000개를 다 보고 센 개수를 알려 준다

---

#### 자세히

**1. 적합성원칙**
- 금융소비자보호법 제17조 ③ — 「… 그 일반금융소비자에게 적합하지 아니하다고 인정되는 계약 체결을 권유해서는 아니 된다」.
- 추천 모델은 모든 고객에게 상품을 권하는 셈이므로 **금융 회사라면 이 기준을 신경 써야 합니다.**

**2. 오늘 쓰는 기준 (이 수업에서 정한 예시)**

| 투자 성향 | 추천해도 되는 위험도 |
|---|---|
| 안정형 | 3 까지 |
| 성장형 | 4 까지 |
| 공격형 | 5 까지 (제한 없음) |

- 실제 기준은 회사마다 다릅니다. 오늘은 이 표를 **지켜야 할 기준**으로 삼습니다.

**3. 어느 추천을 점검하나**
- 오늘 Recall@10 이 가장 높았던 **4주차 행렬분해 추천**(6.1 의 `지난주_추천`)을 점검합니다.
- FM · MLP 는 점수가 더 낮아서 오늘 최종 추천으로 쓰지 않습니다.

In [23]:
# 이 셀에서 하는 일 — 추천 3,000개 중 성향 기준을 넘는 것을 센다
# 왜 하나 — 6.2 에서 U0001 한 명으로 본 문제가 전체로는 얼마나 큰지 잰다

적합_기준 = {"안정형": 3, "성장형": 4, "공격형": 5}    # 딕셔너리 {성향: 추천해도 되는 최대 위험도}

def 안_맞는_개수(추천함수):                        # 추천 함수 자체를 받는다
    """300명의 추천 10개씩을 보고, 성향 기준을 넘는 종목 수를 센다."""
    개수 = 0                                       # 센 개수. 0 부터 시작한다
    for 사람 in users["user_id"]:                  # 300명을 한 명씩
        for 번호 in 추천함수(사람, 이미_담은것.get(사람, set())):   # get = 없으면 빈 집합
            if 위험도[번호] > 적합_기준[성향[사람]]:              # > = 크다. 기준보다 크면 센다
                개수 = 개수 + 1                    # 하나 더 센다
    return 개수                                    # 추천 3,000개 중 기준을 넘은 개수

print(f"4주차 추천 3,000개 중 성향에 맞지 않는 것 : {안_맞는_개수(지난주_추천)}개")

4주차 추천 3,000개 중 성향에 맞지 않는 것 : 205개


**결과 읽는 법**

- 300명에게 10개씩 준 4주차 추천 3,000개 가운데 적합 기준을 넘는 추천의 개수입니다.
  - **205개**가 투자 성향의 기준을 넘었습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.10 위험도 상한 — 후처리 필터링으로 추천을 거르고 상한 정하기</mark>

**6.10 에서 하는 일** — 4주차 추천 순서에서 **상한을 넘는 종목을 빼고**, 남은 종목 중 위에서부터 10개를 추천합니다.

---

> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 성향에 맞지 않는 추천을 없애는 **가장 단순한 방법**입니다. 모델은 그대로 두고, 마지막에 규칙으로 거릅니다.

**왜 중요한가**
- 실무 추천 시스템은 대부분 **「모델이 순위를 매긴다 → 규칙이 거른다」** 두 단계로 되어 있습니다.
  - 법 · 회사 정책 · 재고 같은 규칙을 모델이 스스로 배우게 하는 것은 어렵고, 틀리면 위험합니다
  - 규칙을 따로 두면 기준이 바뀌어도 **모델을 다시 학습하지 않고** 숫자만 바꾸면 됩니다
- 「반드시 지켜야 하는 것」은 모델이 아니라 **규칙**에 맡긴다는 생각이 핵심입니다.

**알아 둘 배경 지식**
- **후처리 필터링 (post-filtering)** — 모델이 만든 목록에서, 나중에 규칙으로 일부를 빼는 것
- **5주차 2단계 추천과의 관계** — 5주차는 「후보 고르기 → 순위 매기기」였습니다. 오늘은 그 뒤에 「거르기」가 하나 더 붙는 모양입니다

**어떻게 계산하나**
1. 4주차 모델로 종목 100개를 점수가 높은 순으로 늘어놓는다
2. 위에서부터 하나씩 보며, 위험도가 그 사람의 **상한 이하인 종목만** 남긴다
3. 남은 목록에서 이미 담은 종목을 빼고 위에서 10개를 추천한다
   - 빠진 자리는 그다음 순위 종목이 자연스럽게 채운다

---

#### 자세히

**1. 모델은 그대로입니다**
- 점수 계산은 4주차와 똑같습니다. **마지막에 걸러 내기만 합니다.**
- 빠진 자리는 **그다음 순위 종목**이 채웁니다. 그래서 추천은 언제나 10개입니다.

**2. 신규 15명도 똑같이 거릅니다**
- 신규 고객은 전체 인기 목록을 받는데, 인기 상위 10개에도 위험도 4 종목이 세 개 있습니다(Taiwan Semiconductor · Invesco QQQ Trust · KODEX 미국나스닥100).
  - 안정형 신규 고객에게는 기준(3 까지)을 넘는 종목입니다.
  - 투자 성향은 가입할 때 받으므로 신규 고객에게도 상한을 걸 수 있습니다.

**3. 상한과 기준을 따로 둔 이유**
- `적합_기준` — **지켜야 할 기준.** 6.9 에서 정했고 바꾸지 않습니다.
- `상한` — **추천 목록에 실제로 거는 값.** 이 절 뒤쪽 「상한 정하기」에서 여러 값으로 바꿔 봅니다.

In [24]:
# 이 셀에서 하는 일 — 4주차 추천 순서에서 상한을 넘는 종목을 빼고 10개를 준다
# 왜 하나 — 모델을 고치지 않고 마지막에 거르기만 해서 성향에 맞지 않는 추천을 없앤다

상한 = {"안정형": 3, "성장형": 4, "공격형": 5}         # 추천 목록에 거는 상한 — 아래 「상한 정하기」에서 바꿔 본다

def 상한_추천(사람, 이미):                               # 4주차 추천 순서에서 상한을 넘는 종목만 빼는 함수
    """4주차 추천 순서에서 성향의 상한을 넘는 종목을 빼고 위에서 10개."""
    순서 = 전체_인기순위                                 # 신규는 인기 목록에서 시작한다
    if 사람 in 사용자_목록:                              # 기존 투자자는 4주차 점수 순서
        순서 = 점수순으로_종목_정렬(사용자_요인[사용자_목록.index(사람)] @ 아이템_요인)   # 6.1 과 같은 계산
    통과 = []                                            # 상한을 통과한 종목을 담을 빈 목록
    for 번호 in 순서:                                    # 점수가 높은 종목부터 하나씩
        if 위험도[번호] <= 상한[성향[사람]]:              # <= 작거나 같다. 상한 이하만 남긴다
            통과.append(번호)                            # 통과하면 목록에 붙인다
    return recsys.take(통과, 이미)                       # 통과한 것 중 이미 담은 것을 빼고 10개

In [25]:
# 이 셀에서 하는 일 — 상한을 건 추천을 채점하고, 성향에 맞지 않는 추천을 다시 센다
# 왜 하나 — 무엇을 얻었고(안 맞는 추천 수) 무엇을 내줬는지(점수) 함께 본다

상한_점수, _ = recsys.recall_at_k(상한_추천, train, test)

print(f"4주차 그대로   — Recall@10 = {지난주_점수:.4f} · 성향에 맞지 않는 추천 {안_맞는_개수(지난주_추천)}개")
print(f"위험도 상한    — Recall@10 = {상한_점수:.4f} · 성향에 맞지 않는 추천 {안_맞는_개수(상한_추천)}개")
print(f"→ 점수 {상한_점수 - 지난주_점수:+.4f}")          # :+.4f = 부호까지 붙여 소수 넷째 자리

4주차 그대로   — Recall@10 = 0.3918 · 성향에 맞지 않는 추천 205개
위험도 상한    — Recall@10 = 0.3899 · 성향에 맞지 않는 추천 0개
→ 점수 -0.0019


**결과 읽는 법**

- 첫 줄은 4주차 추천 그대로입니다.
  - 300명에게 준 추천 3,000개 중 **205개**가 투자 성향의 기준을 넘었습니다.
- 둘째 줄은 같은 추천에 위험도 상한을 건 결과입니다.
  - 기준을 넘는 추천이 **0개**가 되었습니다.
  - Recall@10은 0.3918 → **0.3899**입니다.
- 마지막 줄 `-0.0019`는 상한을 걸어서 **내준 점수**입니다.
  - 205개를 걸러 내는 대가로 Recall@10을 0.0019만큼 잃었다는 뜻입니다.

---

#### 📖 상한 정하기 — 점수와 성향에 맞지 않는 추천 수 비교

**여기서 하는 일** — 상한을 여러 값으로 바꿔 보고, **점수와 성향에 맞지 않는 추천 수**를 나란히 비교합니다.

**1. 무엇을 보고 고를까**
- 상한을 **느슨하게** 걸면 → 점수는 지키지만 기준을 넘는 추천이 남습니다.
- 상한을 **세게** 걸면 → 기준은 지키지만 고객이 실제로 담을 종목까지 빠져 점수가 내려갑니다.

**2. 답할 질문**
- 성향에 맞지 않는 추천을 **0개**로 만들면서 점수를 **가장 덜 내주는** 상한은 무엇일까요?

In [26]:
# 이 셀에서 하는 일 — 상한을 바꿔 가며 점수와 안 맞는 추천 수를 함께 잰다
# 왜 하나 — 기준을 지키는 대가로 점수를 얼마나 내주는지 보고 상한을 고른다

실험 = [{"안정형": 5, "성장형": 5, "공격형": 5},       # 상한 없음 = 4주차
        {"안정형": 4, "성장형": 5, "공격형": 5},       # 안정형의 위험도 5 만 뺀다
        {"안정형": 3, "성장형": 4, "공격형": 5},       # 위의 기본값
        {"안정형": 2, "성장형": 3, "공격형": 4}]       # 기준보다 한 단계씩 더 세게

for 후보 in 실험:                                    # 상한 네 가지를 하나씩
    상한 = 후보                                        # 위 함수가 이 값을 보고 거른다
    점수, _ = recsys.recall_at_k(상한_추천, train, test)   # 위와 같은 채점
    print(f"안정 {후보['안정형']} · 성장 {후보['성장형']} · 공격 {후보['공격형']}   Recall@10 {점수:.4f} · 안 맞는 추천 {안_맞는_개수(상한_추천):3d}개")

상한 = {"안정형": 3, "성장형": 4, "공격형": 5}         # 실험이 끝났으니 기본값으로 되돌린다

안정 5 · 성장 5 · 공격 5   Recall@10 0.3918 · 안 맞는 추천 205개


안정 4 · 성장 5 · 공격 5   Recall@10 0.3862 · 안 맞는 추천 154개
안정 3 · 성장 4 · 공격 5   Recall@10 0.3899 · 안 맞는 추천   0개
안정 2 · 성장 3 · 공격 4   Recall@10 0.2688 · 안 맞는 추천   0개


**결과 읽는 법**

1. **첫 줄** — 상한이 없습니다. 점수는 4주차와 같고, 성향에 맞지 않는 추천이 그대로 남습니다.

2. **둘째 줄** — 안정형에게서 위험도 5 만 뺐습니다. 아직 기준을 넘는 추천이 남습니다.
   - 점수가 셋째 줄보다 조금 낮은 것은 빠진 자리를 채운 종목이 정답이었는지에 따라 점수가 오르내리기 때문입니다. 0.004 정도는 작은 차이입니다.

3. **셋째 줄** — 상한을 기준과 같게 걸었습니다. **안 맞는 추천이 0개**가 되고, 점수는 0.3918 → 0.3899 로 **0.002 만** 내려갑니다.

4. **넷째 줄** — 기준보다 한 단계씩 더 세게 걸었습니다. 안 맞는 추천은 셋째 줄과 똑같이 0개인데 **점수만 더 내려갑니다.**

**한 줄로 정리하면** — 기준보다 세게 걸어도 더 지켜지는 것은 없고 점수만 깎입니다. **적합 기준과 같은 값을 상한으로 거는 것**이 점수를 가장 덜 내주는 선택입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.11 오늘의 마무리</mark>

1. 상품 정보 네 가지를 **원핫 인코딩**으로 0/1 열 41개로 펼쳤고, 사용자 · 종목까지 펼쳐 한 줄에 1 이 6개인 입력을 만들었습니다.
2. **FM** 은 1 인 열들을 둘씩 짝지어 내적을 더하는 모델입니다. 4주차 행렬분해를 모든 짝으로 넓힌 것입니다.
3. **MLP** 는 층을 거치며 계산 모양을 데이터에서 찾는 모델입니다. 층이 많을수록 깊다고 하고, 이것이 딥러닝입니다.
4. 우리 데이터에서는 **행렬분해 0.3918 > FM 0.3463 > MLP 1층 0.3283 > MLP 2층 0.2580** 이었습니다.
   - 복잡한 모델일수록 학습 데이터는 더 잘 맞혔지만, 다음 달 기록은 더 못 맞혔습니다 — **과적합**
   - **Wide&Deep · DeepFM** 은 데이터가 아주 많은 서비스를 위한 모델이라 오늘은 개념만 봤습니다.
5. 4주차 모델은 안정형 U0001 에게 **위험도 4\~5 종목을 네 개** 추천했습니다. 투자 성향별 **위험도 상한**을 걸어 성향에 맞지 않는 추천을 **205개에서 0개**로 줄였고, 그 대가로 점수를 **0.3918 → 0.3899** 로 조금 내줬습니다.

**오늘 기억할 한 줄** — **작은 데이터에서는 단순한 모델이 낫습니다.** 상품 정보는 점수를 올리는 데가 아니라 **투자 성향에 맞지 않는 추천을 거르는 데** 썼습니다.

다음 주에는 지금까지 셀을 하나씩 눌러 한 일 — 모델 학습과 추천 — 을 **사람이 손대지 않아도 한 번에 돌아가도록** 묶고, 결과를 파일로 남깁니다.

In [27]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 점수가 내려간 주차도 그대로 적는다. 점수를 무엇과 바꿨는지 설명할 수 있으면 된다

recsys.record(6, "투자 성향별 위험도 상한", 상한_점수,
              note=f"4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 {안_맞는_개수(지난주_추천)}개 → 0개")

recsys.leaderboard(upto=6)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 6 · 투자 성향별 위험도 상한 · Recall@10 = 0.3899


,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
4,5,2단계 추천 · 후보 생성 + 순위 매기기,0.3382,2주차 세그먼트로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다
5,6,투자 성향별 위험도 상한,0.3899,4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 205개 → 0개


**결과 읽는 법**

- 첫 줄은 오늘 점수를 점수판에 남겼다는 확인입니다.
  - 레벨 6 · 투자 성향별 위험도 상한 · Recall@10 = **0.3899** 입니다.
- 아래 표는 1주차부터 6주차까지의 점수판입니다.
  - `level` 은 주차, `name` 은 그 주의 추천 방식, `recall_at_10` 은 점수, `note` 는 한 줄 설명입니다.
  - `note` 끝의 `...` 은 글이 길어서 화면에서 잘린 것입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다.

답은 바로 아래 **「답」**에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 6.5 결과**

FM 에 상품 정보를 **넣은 것(0.3463)** 이 **넣지 않은 것(0.3670)** 보다 오히려 조금 낮았습니다.

FM 은 상품 정보를 넣을 수 있는 모델인데, 왜 상품 정보가 점수를 올리지 못했을까요?

- 힌트 — 「KODEX 레버리지의 위험도가 5 다」라는 정보는, **종목 번호(`item_id_I079`) 열만 보고도** 알 수 있는 정보일까요? 모델은 종목 번호 열의 잠재 요인을 거래 기록에서 이미 배웁니다.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 2 · 6.10 결과**

위험도 상한으로 성향에 맞지 않는 추천을 **205개 → 0개**로 줄였는데, Recall@10 은 0.3918 → 0.3899 로 **0.0019 만** 내려갔습니다.

추천 목록에서 종목이 많이 빠졌는데, 왜 점수는 거의 그대로일까요?

- 힌트 — 6.10 의 「빠진 자리는 그다음 순위 종목이 채웁니다」를 다시 읽어 보세요. 빠진 종목이 **원래 정답이었는지**가 열쇠입니다.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 3 · 6.10 결과**

회사가 기준을 바꿔 **「안정형은 위험도 2 까지」**로 더 엄격하게 정했다고 해 봅시다.

지금 상한(안정형 3 · 성장형 4 · 공격형 5)을 그대로 두면 어떻게 될까요? 상한을 어떻게 바꿔야 하고, 그 대가는 얼마일까요?

- 힌트 — 6.10 「상한 정하기」 표에서 **상한 숫자와 「안 맞는 추천」 개수**가 어떻게 함께 움직이는지 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북을 실제로 실행해서 얻은 값입니다.

**함께 보기 답 — 과적합입니다. 학습 데이터를 외워서 처음 보는 기록을 못 맞혔습니다**

- MLP 2층이 학습할 숫자는 **약 2만 9천 개**입니다.
  - 학습 데이터에서 「담았다(1)」는 **7,095줄**뿐입니다. 정할 숫자가 배울 예시보다 4배쯤 많습니다.
- 숫자가 이렇게 많으면 모델은 「어떤 사람이 어떤 종목을 왜 담는가」라는 **규칙을 찾지 않아도** 됩니다.
  - 학습 데이터의 줄 하나하나를 **그대로 외워도** 정답을 맞힐 수 있기 때문입니다.
  - 그래서 학습 데이터 정답률은 **0.9785** 까지 올라갔습니다.
- 외운 것은 다음 달 기록에 쓸 수 없습니다. 다음 달에 담을 종목은 학습 데이터에 없던 짝이기 때문입니다.
  - 그래서 Recall@10 은 **0.2580** 으로 가장 낮았습니다.
- 이것이 **과적합(overfitting)** 입니다. 4주차에서 잠재 요인을 64개로 늘렸을 때 점수가 **0.1778** 로 떨어진 것도 같은 이유였습니다.
- **한 줄로 정리하면** — 데이터가 작을 때 모델을 복잡하게 만들면 규칙 대신 데이터를 외웁니다. **작은 데이터에서는 단순한 모델이 낫습니다.**

**혼자 해 보기 1 답 — 상품 정보는 종목 번호만 알면 정해지는 정보라서, 모델이 거래 기록에서 이미 배운 것이었기 때문입니다**

- 종목 하나에 섹터 · 자산군 · 위험도 · 테마는 **하나씩 정해져 있습니다.**
  - `item_id_I079` 열이 1 이면 `risk_level_5` 열도 **언제나** 1 입니다.
  - 그래서 상품 정보 열은 종목 번호 열이 이미 가진 정보를 **다시 알려 줄 뿐**, 새 정보를 주지 않습니다.
- 종목 번호 열의 잠재 요인은 거래 기록에서 학습됩니다.
  - 「KODEX 레버리지를 담는 사람들이 어떤 사람들인가」를 배우면서, 위험도 5 라는 성격도 함께 배웁니다.
- 반대로 상품 정보 열 41개를 넣으면 **학습할 숫자가 늘어납니다**(열 385개 → 426개).
  - 새 정보 없이 숫자만 늘어서 점수가 조금 내려갔습니다. 6.7 의 「복잡할수록 낮아진다」와 같은 방향입니다.
  - 다만 차이(0.02)는 씨앗을 바꾸면 달라질 정도로 작습니다. 확실한 것은 **「넣어도 오르지 않았다」**입니다.
- 상품 정보가 도움이 되는 경우는 따로 있습니다 — **거래 기록이 거의 없는 새 종목**입니다. 그 종목의 번호 열은 배울 기록이 없지만, 위험도 · 섹터 열은 다른 종목의 기록으로 이미 배워 두었습니다.
- **한 줄로 정리하면** — 거래 기록이 충분한 종목에는 상품 정보가 새로 알려 주는 것이 없습니다.

**혼자 해 보기 2 답 — 빠진 종목 대부분이 원래 정답이 아니었고, 빈자리를 채운 종목이 그만큼 다시 맞혔기 때문입니다**

- 채점 대상 266명의 추천에서 상한 때문에 빠진 종목은 **179개**입니다.
  - 그중 실제로 다음 달에 담은 정답은 **15개**뿐입니다.
- 빠진 자리에는 그다음 순위 종목 **179개**가 새로 들어왔습니다.
  - 그중 정답도 **15개**입니다. 잃은 만큼 다시 맞힌 셈입니다.
- 맞힌 개수가 같은데도 점수가 0.0019 내려간 것은 Recall@10 의 계산 방식 때문입니다.
  - Recall@10 은 사람마다 「맞힌 개수 ÷ 그 사람의 정답 개수」를 구한 뒤 평균을 냅니다.
  - 그래서 같은 한 개라도 사람마다 점수에 주는 무게가 다르고, 잃은 쪽의 무게가 조금 더 컸습니다.
- **한 줄로 정리하면** — 성향에 맞지 않는 추천은 대부분 어차피 맞히지 못하는 추천이었고, 그래서 걸러 내도 점수를 거의 잃지 않았습니다.

**혼자 해 보기 3 답 — 지금 상한으로는 324개가 새 기준을 넘고, 상한을 2 · 4 · 5 로 낮추면 0개가 되는 대신 점수가 0.3704 로 내려갑니다**

- 상한을 그대로(안정형 3) 두면, 안정형에게 간 위험도 3 종목이 모두 새 기준을 넘게 됩니다.
  - 실제로 세어 보면 성향에 맞지 않는 추천이 **324개**가 됩니다.
- 상한도 기준에 맞춰 **안정형 2 · 성장형 4 · 공격형 5** 로 바꿔야 합니다.
  - 그러면 안 맞는 추천은 다시 **0개**가 됩니다.
  - 대신 Recall@10 이 0.3899 → **0.3704** 로 0.0195 내려갑니다. 6.8 의 실험에서 본 것처럼, 상한이 엄격할수록 고객이 실제로 담을 종목도 함께 빠지기 때문입니다.
- **한 줄로 정리하면** — 기준이 바뀌면 상한도 같이 바꿔야 하고, 기준이 엄격할수록 점수를 더 많이 내줍니다.